# Авито: воспроизведение отправленного решения

**Цель:** для каждого поискового запроса найти до 50 объявлений услуг из корпуса.
Решение применяет каскад: широкий лексический и семантический поиск →
CatBoost до 1500 → CatBoost до 750 → LambdaMART и дообученный
BGE-reranker-v2-m3 до 50. Ранние этапы берегут полноту, а дорогая текстовая
модель оценивает только сокращённый список. Финальные баллы LambdaMART и BGE
объединяются фиксированной смесью рангов.

**Подробности:** прочитайте соседний файл
[SOLUTION_OVERVIEW_0850967.md](SOLUTION_OVERVIEW_0850967.md). Его полная копия
приложена в конце ноутбука, если ноутбук открыт без остальных файлов проекта.
В документе описаны признаки, модели, локальные метрики и разбор ошибок.

**Запуск:** прикрепите Kaggle Dataset с `train.parquet`,
`benchmark_queries.parquet` и `benchmark_items.parquet`; включите
**GPU T4 × 2** и доступ к весам `BAAI/bge-m3` и
`BAAI/bge-reranker-v2-m3` через Internet On либо прикреплённые модели.
Начните с чистой Kaggle-сессии и выполняйте ячейки сверху вниз.
Полный прогон занимает ориентировочно
**6 часов** на двух T4; время зависит от сессии Kaggle и кеша.

**Отправленная конфигурация:** BGE cross-encoder дообучается 3 эпохи,
максимум 192 токена на пару.
Последняя ячейка записывает `/kaggle/working/answer.csv` и проверяет его
формат. Проверяющая система оценила отправленный файл в **0,850967**
Recall@50.


## 1. Данные и разделение

Сначала читаем Parquet и фиксируем seed. Проверочные тексты запросов исключаются из обучения соответствующих supervised-моделей.


In [ ]:
# Фиксируем конфигурацию и зерно случайности отправленной версии.
import gc, json, math, os, random, re, sqlite3, time
from collections import Counter, defaultdict
from functools import lru_cache
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import scipy.sparse as sp
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize as l2_normalize
import joblib

PREVIOUS_SPLIT_SEED = 20260925
SECOND_SPLIT_SEED = 20260926
THIRD_SPLIT_SEED = 20260927
FOURTH_SPLIT_SEED = 20260928
SEED = 20260929
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
WORK = Path("/kaggle/working/avito_passage_memory_750_v1") if Path("/kaggle").exists() else Path.cwd() / "artifacts" / "kaggle_passage_memory_750_v1"
WORK.mkdir(parents=True, exist_ok=True)

# Пятое текстовое разбиение: 1200 development + 500 закрытый holdout.
N_DEV, N_HOLDOUT = 1200, 500
POOL_LIMIT, LEXICAL_RESERVE, PREFIX_PER_EXPRESSION = 10_000, 2_000, 400
VECTOR_GLOBAL_K, VECTOR_NEAR_K, NEAR_RADIUS_KM = 200, 500, 100
LINEAR_DIM, LINEAR_EPOCHS, LINEAR_BATCH = 128, 4, 256
TFIDF_MAX_FEATURES, LSA_DIM = 60_000, 64
BGE_MAX_TOKENS, BGE_BATCH, BGE_CHUNK = 192, 24, 768
BGE_MODEL_SOURCE = None  # Путь к прикреплённой модели или None: искать локально/скачать BAAI/bge-m3.
print("Device:", DEVICE, "GPU:", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "нет")

In [ ]:
# Находим три Parquet-файла и строим разбиение по текстам запросов.
"""Fifth text-disjoint split; reconstruct and exclude the viewed fourth split."""


def find_data_dir():
    override = os.environ.get("AVITO_DATA_DIR")
    if override:
        root = Path(override)
        assert all((root / name).exists() for name in (
            "train.parquet", "benchmark_items.parquet", "benchmark_queries.parquet"))
        return root
    matches = [p.parent for p in Path("/kaggle/input").rglob("train.parquet")
               if (p.parent / "benchmark_items.parquet").exists()
               and (p.parent / "benchmark_queries.parquet").exists()]
    if len(matches) != 1:
        raise FileNotFoundError(f"Ожидалась одна папка с тремя Parquet: {matches}")
    return matches[0]


DATA = find_data_dir()
ITEM_COLS = ["item_id", "item_title_raw", "item_description_raw",
             "item_infm_params_text", "item_location_id", "item_category_id",
             "item_microcat_id", "item_price", "item_rating",
             "item_rating_reviews_count", "item_latitude", "item_longitude"]
Q_COLS = ["search_query", "search_location_id", "search_is_delivery_search",
          "search_infm_params_text", "search_category"]
items = pd.read_parquet(DATA / "benchmark_items.parquet", columns=ITEM_COLS)
assert items.item_id.is_unique
item_to_pos = {item_id: pos for pos, item_id in enumerate(items.item_id)}
N_ITEMS = len(items)
print("Dataset:", DATA, "| items:", N_ITEMS,
      "| train rows:", pq.ParquetFile(DATA / "train.parquet").metadata.num_rows)

# Только известные выбранные объявления, которые есть в корпусе.
events = defaultdict(set)
for batch in pq.ParquetFile(DATA / "train.parquet").iter_batches(
        batch_size=20_000, columns=Q_COLS + ["item_id"]):
    rows = batch.to_pydict()
    for row, item_id in enumerate(rows["item_id"]):
        pos = item_to_pos.get(item_id)
        if pos is not None:
            events[tuple(rows[column][row] for column in Q_COLS)].add(pos)
by_text = defaultdict(list)
for event in events:
    by_text[event[0]].append(event)

all_texts = sorted(by_text, key=lambda value: str(value))
first_order = all_texts.copy()
random.Random(PREVIOUS_SPLIT_SEED).shuffle(first_order)
first_inspected = set(first_order[:1000])

after_first = [text for text in all_texts if text not in first_inspected]
second_order = after_first.copy()
random.Random(SECOND_SPLIT_SEED).shuffle(second_order)
second_inspected = set(second_order[:2000])

after_second = [text for text in all_texts
                if text not in first_inspected and text not in second_inspected]
third_order = after_second.copy()
random.Random(THIRD_SPLIT_SEED).shuffle(third_order)
third_inspected = set(third_order[:2000])


def split_bucket(query):
    words = len(("" if query.get("search_query") is None
                 else str(query["search_query"])).split())
    length = "1-2" if words <= 2 else ("3" if words == 3 else "4+")
    filter_text = query.get("search_infm_params_text")
    return length, bool("" if filter_text is None else str(filter_text).strip())


# Benchmark содержит только признаки; его неизвестные ответы не читаются.
benchmark_features = pd.read_parquet(
    DATA / "benchmark_queries.parquet",
    columns=["search_query", "search_infm_params_text", "search_category"])
benchmark_service = benchmark_features[benchmark_features.search_category == 114]
target_counts = Counter(split_bucket(row)
                        for row in benchmark_service.to_dict("records"))


def matched_quotas(total):
    scaled = {bucket: total * count / len(benchmark_service)
              for bucket, count in target_counts.items()}
    result = {bucket: int(math.floor(value)) for bucket, value in scaled.items()}
    remainder = total - sum(result.values())
    order = sorted(scaled, key=lambda bucket: (-(scaled[bucket] - result[bucket]),
                                               bucket))
    for bucket in order[:remainder]:
        result[bucket] += 1
    return result


def choose_split(texts, seed, dev_total=N_DEV, holdout_total=N_HOLDOUT):
    """Same deterministic procedure used by the previous notebook."""
    dev_quotas = matched_quotas(dev_total)
    holdout_quotas = matched_quotas(holdout_total)
    order = list(texts)
    rng = random.Random(seed)
    rng.shuffle(order)
    chosen_event = {}
    buckets = defaultdict(list)
    for text in order:
        service = [event for event in by_text[text] if event[-1] == 114]
        event = rng.choice(service or by_text[text])
        chosen_event[text] = event
        buckets[split_bucket(dict(zip(Q_COLS, event)))].append(text)
    assert all(len(buckets[bucket]) >= holdout_quotas[bucket] + dev_quotas[bucket]
               for bucket in target_counts)
    holdout = [text for bucket in sorted(target_counts)
               for text in buckets[bucket][:holdout_quotas[bucket]]]
    development = [text for bucket in sorted(target_counts)
                   for text in buckets[bucket][
                       holdout_quotas[bucket]:
                       holdout_quotas[bucket] + dev_quotas[bucket]]]
    return development, holdout, chosen_event


after_third = [text for text in all_texts if text not in (
    first_inspected | second_inspected | third_inspected)]
old_dev, old_holdout, _ = choose_split(
    after_third, FOURTH_SPLIT_SEED, dev_total=1500, holdout_total=500)
fourth_inspected = set(old_dev + old_holdout)
assert len(fourth_inspected) == 2000

prior_inspected = (first_inspected | second_inspected |
                   third_inspected | fourth_inspected)
available = [text for text in all_texts if text not in prior_inspected]
development_texts, holdout_texts, candidate_event = choose_split(available, SEED)
selected_texts = development_texts + holdout_texts
selected = [candidate_event[text] for text in selected_texts]
cases = [{"query": dict(zip(Q_COLS, key)), "gold": events[key]}
         for key in selected]
excluded_texts = set(selected_texts)
assert len(excluded_texts) == len(cases) == N_DEV + N_HOLDOUT
assert excluded_texts.isdisjoint(prior_inspected)
assert set(development_texts).isdisjoint(holdout_texts)
print("Пятое разбиение:", N_DEV, "development /", N_HOLDOUT,
      "holdout | прежних просмотренных текстов:", len(prior_inspected))
print("Development-страты:",
      Counter(split_bucket(case["query"]) for case in cases[:N_DEV]))
print("Holdout-страты:",
      Counter(split_bucket(case["query"]) for case in cases[N_DEV:]))
print("Ограничение: category=0 представлена слишком слабо для надёжной оценки.")


## 2. Широкий поиск

Объединяем лексический поиск, географию и независимые векторные источники. Этот этап стремится сохранить выбранное объявление в пуле порядка 10 тысяч.


In [ ]:
# Лексический и географический источники широкого пула.
WORD_RE = re.compile(r"[a-zа-я0-9]+", re.IGNORECASE)
RU_SUFFIXES = tuple(sorted({
    "иями", "ями", "ами", "иях", "иям", "ией", "остью", "ости", "ость",
    "ыми", "ими", "ого", "ему", "ому", "ее", "ие", "ые", "ое", "ая", "яя",
    "ый", "ий", "ой", "ую", "юю", "ых", "их", "ым", "им", "ов", "ев",
    "ей", "ам", "ям", "ах", "ях", "ом", "ем", "ою", "ею", "ия", "ии", "ию",
    "ат", "ят", "ет", "ют", "ут", "ит", "ишь", "ить", "ать", "ять",
    "ение", "ения", "ений", "енный", "енная", "анные", "ание", "ания",
    "а", "я", "ы", "и", "е", "о", "у", "ю"}, key=len, reverse=True))
STOPWORDS = {"и", "в", "во", "на", "по", "за", "из", "для", "от", "до", "с", "со",
             "к", "у", "а", "или", "это", "мне", "нужно", "ищу", "услуги", "услуга"}

@lru_cache(maxsize=150_000)
def stem(word):
    if word.isascii() or len(word) < 5: return word
    for suffix in RU_SUFFIXES:
        if word.endswith(suffix) and len(word) - len(suffix) >= 4:
            return word[:-len(suffix)]
    return word

def tokens(value, limit=None):
    text = "" if value is None or pd.isna(value) else str(value)
    if limit is not None: text = text[:limit]
    return [stem(w) for w in WORD_RE.findall(text.lower().replace("ё", "е")) if w not in STOPWORDS]

def norm(value, limit=None): return " ".join(tokens(value, limit))

def as_int(value, missing=-1): return missing if value is None or pd.isna(value) else int(value)

locs = np.asarray([as_int(x) for x in items.item_location_id], dtype=np.int64)
microcats = np.asarray([as_int(x) for x in items.item_microcat_id], dtype=np.int64)
reviews = pd.to_numeric(items.item_rating_reviews_count, errors="coerce").fillna(0).clip(lower=0).to_numpy(np.float32)
centers = {}
geo = items[["item_location_id", "item_latitude", "item_longitude"]].dropna()
for loc, row in geo.groupby("item_location_id")[["item_latitude", "item_longitude"]].mean().iterrows():
    centers[int(loc)] = (float(row.item_latitude), float(row.item_longitude))

def distance_km(a, b):
    p1, p2 = math.radians(a[0]), math.radians(b[0])
    dp, dl = p2-p1, math.radians(b[1]-a[1])
    x = math.sin(dp/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return 12742.0 * math.asin(min(1.0, math.sqrt(x)))

by_location = {}
for loc in np.unique(locs):
    idx = np.flatnonzero(locs == loc)
    by_location[int(loc)] = idx[np.argsort(-reviews[idx], kind="stable")]
near_cache = {}
def nearby_locations(location):
    location = as_int(location)
    if location not in near_cache:
        center = centers.get(location)
        near_cache[location] = tuple(loc for loc, point in centers.items()
                                     if center is not None and distance_km(center, point) <= NEAR_RADIUS_KM)
    return near_cache[location]

near_indices_cache = {}
def nearby_indices(location):
    location = as_int(location)
    if location not in near_indices_cache:
        near = nearby_locations(location)
        near_indices_cache[location] = (
            np.concatenate([by_location[loc] for loc in near]).astype(np.int32)
            if near else np.empty(0, np.int32))
    return near_indices_cache[location]

DB = WORK / "fts5_items.sqlite"
def open_fts():
    con = sqlite3.connect(DB)
    con.execute("PRAGMA cache_size=-48000")
    con.execute("PRAGMA temp_store=MEMORY")
    return con

def build_fts():
    if DB.exists():
        try:
            existing = open_fts()
            count = existing.execute("SELECT COUNT(*) FROM item_loc").fetchone()[0]
            existing.close()
            if count == N_ITEMS:
                print("Используется индекс FTS5:", DB); return
        except sqlite3.DatabaseError: pass
        DB.unlink()
    con = open_fts()
    con.execute("PRAGMA journal_mode=OFF")
    con.execute("PRAGMA synchronous=OFF")
    con.execute("CREATE TABLE item_loc(rowid INTEGER PRIMARY KEY, location_id INTEGER)")
    con.execute("CREATE VIRTUAL TABLE fts USING fts5(title, params, description, tokenize='unicode61')")
    for start in range(0, N_ITEMS, 1000):
        end = min(start+1000, N_ITEMS)
        con.executemany("INSERT INTO item_loc VALUES (?,?)",
                        ((j+1, int(locs[j])) for j in range(start, end)))
        con.executemany("INSERT INTO fts(rowid,title,params,description) VALUES (?,?,?,?)",
                        ((j+1, norm(items.item_title_raw.iat[j], 240),
                          norm(items.item_infm_params_text.iat[j], 1200),
                          norm(items.item_description_raw.iat[j], 3500))
                         for j in range(start, end)))
        con.commit()
        if end % 20_000 == 0 or end == N_ITEMS: print("FTS:", end, "/", N_ITEMS)
    con.execute("CREATE INDEX ix_location ON item_loc(location_id)")
    con.commit(); con.close()

build_fts()
con = open_fts()

def fts_hits(expression, location=None, limit=10_000):
    where = " AND item_loc.location_id=?" if location is not None else ""
    sql = ("SELECT fts.rowid-1, -bm25(fts,8.0,1.3,1.0) FROM fts "
           "JOIN item_loc ON item_loc.rowid=fts.rowid WHERE fts MATCH ?" + where +
           " ORDER BY bm25(fts,8.0,1.3,1.0) LIMIT ?")
    args = (expression, as_int(location), limit) if location is not None else (expression, limit)
    return con.execute(sql, args).fetchall()

def lexical_hits(query):
    terms = list(dict.fromkeys(tokens(query["search_query"])))
    if not terms: return {}
    quoted = ['"'+w+'"' for w in terms]
    expressions = [" OR ".join(quoted)]
    if len(terms) > 1: expressions.append(" AND ".join(quoted))
    hits = {}
    for expression in expressions:
        for location in (query.get("search_location_id"), None):
            for pos, score in fts_hits(expression, location):
                hits[pos] = max(score, hits.get(pos, -float("inf")))
    return hits

def prefix_expressions(query):
    words = sorted({w for w in tokens(query["search_query"]) if len(w) >= 5},
                   key=lambda w: (-len(w), w))[:4]
    out = []
    for word in words:
        out.append('"'+word[:4]+'"*')
        if len(word) >= 8: out.append('"'+word[:6]+'"*')
    return tuple(dict.fromkeys(out))

@lru_cache(maxsize=4096)
def prefix_hits(expression, near):
    if not near:
        return [pos for pos, _ in fts_hits(expression, None, PREFIX_PER_EXPRESSION)]
    placeholders = ",".join("?" for _ in near)
    sql = ("SELECT fts.rowid-1 FROM fts JOIN item_loc ON item_loc.rowid=fts.rowid "
           f"WHERE fts MATCH ? AND item_loc.location_id IN ({placeholders}) "
           "ORDER BY bm25(fts,8.0,1.3,1.0) LIMIT ?")
    return [row[0] for row in con.execute(sql, (expression, *near, PREFIX_PER_EXPRESSION))]

def make_broad_pool(query):
    lex_scores = lexical_hits(query)
    lex = sorted(lex_scores, key=lex_scores.get, reverse=True)
    local = by_location.get(as_int(query.get("search_location_id")), np.empty(0, np.int32))
    if len(local) <= POOL_LIMIT - LEXICAL_RESERVE:
        sequence = list(local) + lex
    else:
        sequence = lex[:LEXICAL_RESERVE] + list(local)
    broad = list(dict.fromkeys(int(i) for i in sequence))[:POOL_LIMIT]
    seen = set(broad)
    prefixes = {}
    near = nearby_locations(query.get("search_location_id"))
    for expression in prefix_expressions(query):
        for rank, pos in enumerate(prefix_hits(expression, near), 1):
            prefixes[pos] = min(rank, prefixes.get(pos, rank))
            if pos not in seen: broad.append(pos); seen.add(pos)
    return np.asarray(broad, dtype=np.int32), np.asarray(lex, dtype=np.int32), prefixes

pool_cache = WORK / f"validation_broad_pools_{N_DEV+N_HOLDOUT}_{POOL_LIMIT}_{N_ITEMS}.joblib"
if pool_cache.exists():
    saved = joblib.load(pool_cache)
    assert len(saved) == len(cases)
    for case, (base, lex, prefixes) in zip(cases, saved):
        case.update(base=base, lex=lex, prefixes=prefixes)
else:
    saved = []
    for n, case in enumerate(cases, 1):
        base, lex, prefixes = make_broad_pool(case["query"])
        case.update(base=base, lex=lex, prefixes=prefixes)
        saved.append((base, lex, prefixes))
        if n % 100 == 0: print("Пулы:", n, "/", len(cases), flush=True)
    joblib.dump(saved, pool_cache, compress=3)

def recall_of(cases_subset, field):
    return float(np.mean([len(c["gold"].intersection(c[field]))/len(c["gold"])
                          for c in cases_subset]))
print("Широкие пулы построены для", len(cases), "запросов.")


In [ ]:
# TF-IDF и LSA: дешёвый независимый семантический источник.
def query_text(query):
    # Поисковый текст и фильтр можно читать обеим семантическим моделям.
    return norm(query.get("search_query")) + " " + norm(query.get("search_infm_params_text"))

def item_text(title, params, description):
    # Заголовок повторяется: он обычно точнее длинного рекламного описания.
    t = norm(title, 240)
    return (t + " ") * 3 + norm(params, 800) + " " + norm(description, 1600)

corpus_docs = [item_text(t, p, d) for t, p, d in zip(
    items.item_title_raw, items.item_infm_params_text, items.item_description_raw)]
val_q_docs = [query_text(c["query"]) for c in cases]

# Сохраняем один текст на item_id, но все обучающие выбранные пары.
# Группировка по исходному поисковому тексту позволяет брать не более одной
# пары с одним запросом в mini-batch и сокращает ложные in-batch negatives.
train_item_to_idx, train_item_docs, train_item_microcats = {}, [], []
train_q_to_idx, train_q_docs = {}, []
pairs_by_query_text = defaultdict(list)
label_counts = Counter()
term_label_counts = defaultdict(Counter)
GENERIC_FILTER = {"вид", "тип", "услуг", "рейтинг", "пользовател", "звезд", "выше"}

def prior_terms(q, f):
    return set(tokens(q)) | {"filter:"+w for w in set(tokens(f)) - GENERIC_FILTER}

train_cols = ["search_query", "search_infm_params_text", "item_id", "item_title_raw",
              "item_infm_params_text", "item_description_raw", "item_microcat_id"]
excluded_normalized = {norm(text) for text in excluded_texts}
excluded_count = 0
for batch in pq.ParquetFile(DATA / "train.parquet").iter_batches(batch_size=10_000, columns=train_cols):
    d = batch.to_pydict()
    for j in range(batch.num_rows):
        raw_q = d["search_query"][j]
        if raw_q in excluded_texts or norm(raw_q) in excluded_normalized:
            excluded_count += 1
            continue
        qdoc = query_text({"search_query": raw_q, "search_infm_params_text": d["search_infm_params_text"][j]})
        iid = d["item_id"][j]
        if not qdoc or iid is None: continue
        if qdoc not in train_q_to_idx:
            train_q_to_idx[qdoc] = len(train_q_docs); train_q_docs.append(qdoc)
        if iid not in train_item_to_idx:
            train_item_to_idx[iid] = len(train_item_docs)
            train_item_docs.append(item_text(d["item_title_raw"][j],
                                             d["item_infm_params_text"][j],
                                             d["item_description_raw"][j]))
            raw_label = d["item_microcat_id"][j]
            train_item_microcats.append(
                int(raw_label) if raw_label is not None else -1)
        q_idx, i_idx = train_q_to_idx[qdoc], train_item_to_idx[iid]
        pairs_by_query_text[raw_q].append((q_idx, i_idx))
        label = d["item_microcat_id"][j]
        if label is not None:
            label = int(label)
            label_counts[label] += 1
            for term in prior_terms(raw_q, d["search_infm_params_text"][j]):
                term_label_counts[term][label] += 1

assert not (set(pairs_by_query_text) & excluded_texts)
groups = list(pairs_by_query_text.values())
print("Обучающие тексты запросов:", len(groups), "| пары:", sum(map(len, groups)),
      "| train item_id:", len(train_item_docs), "| исключено строк:", excluded_count)

tfidf_path = WORK / "tfidf.joblib"
if tfidf_path.exists():
    tfidf = joblib.load(tfidf_path)
else:
    tfidf = TfidfVectorizer(max_features=TFIDF_MAX_FEATURES, min_df=2,
                            sublinear_tf=True, dtype=np.float32)
    from itertools import chain
    tfidf.fit(chain(corpus_docs, train_item_docs, train_q_docs))
    joblib.dump(tfidf, tfidf_path, compress=3)
print("Размер словаря:", len(tfidf.vocabulary_))

def cached_csr(name, texts):
    path = WORK / f"{name}.npz"
    if path.exists(): return sp.load_npz(path)
    x = tfidf.transform(texts).tocsr()
    sp.save_npz(path, x, compressed=True)
    return x

X_items = cached_csr("tfidf_corpus", corpus_docs)
X_train_q = cached_csr("tfidf_train_q", train_q_docs)
X_train_i = cached_csr("tfidf_train_i", train_item_docs)
X_val_q = tfidf.transform(val_q_docs).tocsr()
assert X_items.shape[0] == N_ITEMS
assert X_train_q.shape[0] == len(train_q_docs)
assert X_train_i.shape[0] == len(train_item_docs)
print("TF–IDF matrices:", X_items.shape, X_train_q.shape, X_train_i.shape)

lsa_path = WORK / "lsa.joblib"
lsa_item_path = WORK / "lsa_items.npy"
if lsa_path.exists() and lsa_item_path.exists():
    lsa = joblib.load(lsa_path)
    lsa_items = np.load(lsa_item_path, mmap_mode="r")
else:
    lsa = TruncatedSVD(n_components=LSA_DIM, n_iter=5, random_state=SEED)
    lsa_items = l2_normalize(lsa.fit_transform(X_items).astype(np.float32))
    joblib.dump(lsa, lsa_path, compress=3)
    np.save(lsa_item_path, lsa_items)
lsa_queries = l2_normalize(lsa.transform(X_val_q).astype(np.float32))
print("LSA:", lsa_items.shape, lsa_queries.shape)

In [ ]:
# Линейные векторы запроса и объявления, обученные на выбранных парах.
class SparseLinearTower(nn.Module):
    def __init__(self, vocab_size, dim):
        super().__init__()
        # Явно задаём float32: в общей Kaggle-сессии default dtype может
        # остаться float16 после других опытов с нейросетями.
        self.query = nn.EmbeddingBag(vocab_size, dim, mode="sum", dtype=torch.float32)
        self.item = nn.EmbeddingBag(vocab_size, dim, mode="sum", dtype=torch.float32)
        nn.init.normal_(self.query.weight, std=0.02)
        nn.init.normal_(self.item.weight, std=0.02)

    def encode(self, x, side):
        # CSR batch -> EmbeddingBag: это эффективное X @ W без densify(X).
        indptr = torch.as_tensor(x.indptr[:-1], dtype=torch.long, device=DEVICE)
        indices = torch.as_tensor(x.indices.astype(np.int64, copy=False), device=DEVICE)
        table = getattr(self, side)
        # EmbeddingBag требует одинаковый dtype таблицы и per_sample_weights.
        weights = torch.as_tensor(x.data, dtype=table.weight.dtype, device=DEVICE)
        raw = table(indices, indptr, per_sample_weights=weights)
        return F.normalize(raw.float(), p=2, dim=1)

model = SparseLinearTower(X_items.shape[1], LINEAR_DIM).to(
    device=DEVICE, dtype=torch.float32)
linear_state = WORK / "linear_towers.pt"
if linear_state.exists():
    model.load_state_dict(torch.load(linear_state, map_location=DEVICE, weights_only=True))
    print("Загружены линейные башни из", linear_state)
else:
    assert model.query.weight.dtype == model.item.weight.dtype == torch.float32
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    group_order = np.arange(len(groups))
    train_rng = np.random.default_rng(SEED)
    for epoch in range(LINEAR_EPOCHS):
        train_rng.shuffle(group_order)
        epoch_losses = []
        bq, bi, used_items, used_queries = [], [], set(), set()

        def update_batch():
            if len(bq) < 2: return None
            model.train(); optimizer.zero_grad(set_to_none=True)
            qv = model.encode(X_train_q[np.asarray(bq)], "query")
            iv = model.encode(X_train_i[np.asarray(bi)], "item")
            logits = (qv @ iv.T) / 0.07
            target = torch.arange(len(bq), device=DEVICE)
            loss = 0.5 * (F.cross_entropy(logits, target) +
                          F.cross_entropy(logits.T, target))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            return float(loss.detach().cpu())

        for g in group_order:
            candidate_pairs = groups[int(g)]
            q_idx, i_idx = candidate_pairs[int(train_rng.integers(len(candidate_pairs)))]
            if i_idx in used_items or q_idx in used_queries: continue
            bq.append(q_idx); bi.append(i_idx)
            used_items.add(i_idx); used_queries.add(q_idx)
            if len(bq) >= LINEAR_BATCH:
                value = update_batch()
                if value is not None: epoch_losses.append(value)
                bq, bi, used_items, used_queries = [], [], set(), set()
        value = update_batch()
        if value is not None: epoch_losses.append(value)
        print(f"Epoch {epoch+1}/{LINEAR_EPOCHS}: mean loss={np.mean(epoch_losses):.4f}", flush=True)
    torch.save(model.state_dict(), linear_state)

@torch.no_grad()
def encode_csr(x, side, batch_size=2048):
    model.eval()
    result = np.empty((x.shape[0], LINEAR_DIM), np.float32)
    for start in range(0, x.shape[0], batch_size):
        end = min(start+batch_size, x.shape[0])
        result[start:end] = model.encode(x[start:end], side).cpu().numpy()
    return result

linear_item_path = WORK / "linear_items.npy"
if linear_item_path.exists():
    linear_items = np.load(linear_item_path, mmap_mode="r")
else:
    linear_items = encode_csr(X_items, "item")
    np.save(linear_item_path, linear_items)
linear_queries = encode_csr(X_val_q, "query")
print("Linear vectors:", linear_items.shape, linear_queries.shape)

In [ ]:
# Поиск ближайших объявлений по векторным источникам.
@torch.no_grad()
def retrieve_vector_sources(item_vectors, query_vectors, source_name):
    # При стандартном Kaggle GPU весь массив 189k x 64/128 помещается в VRAM.
    iv = torch.as_tensor(np.asarray(item_vectors), dtype=torch.float32, device=DEVICE)
    result = []
    for n, (case, q_np) in enumerate(zip(cases, query_vectors), 1):
        if not np.any(q_np):
            result.append(np.empty(0, np.int32)); continue
        qv = torch.as_tensor(q_np, dtype=torch.float32, device=DEVICE)
        scores = iv @ qv
        chosen = []
        if VECTOR_GLOBAL_K:
            chosen.extend(torch.topk(scores, k=min(VECTOR_GLOBAL_K, N_ITEMS)).indices.cpu().tolist())
        near = nearby_indices(case["query"].get("search_location_id"))
        if len(near):
            near_t = torch.as_tensor(near.astype(np.int64), device=DEVICE)
            pos = torch.topk(scores[near_t], k=min(VECTOR_NEAR_K, len(near))).indices
            chosen.extend(near_t[pos].cpu().tolist())
        unique = list(dict.fromkeys(chosen))
        if unique:
            rank = torch.argsort(scores[torch.as_tensor(unique, device=DEVICE)], descending=True).cpu().numpy()
            unique = np.asarray(unique, np.int32)[rank]
        result.append(np.asarray(unique, np.int32))
        if n % 250 == 0: print(source_name, n, "/", len(cases), flush=True)
    del iv
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
    return result

lsa_sources = retrieve_vector_sources(lsa_items, lsa_queries, "LSA")
linear_sources = retrieve_vector_sources(linear_items, linear_queries, "linear")
for case, lsa_ids, lin_ids in zip(cases, lsa_sources, linear_sources):
    case["lsa"] = lsa_ids; case["linear"] = lin_ids
    case["pool_lsa"] = np.unique(np.concatenate((case["base"], lsa_ids)))
    case["pool_linear"] = np.unique(np.concatenate((case["base"], lin_ids)))
    case["pool_both"] = np.unique(np.concatenate((case["base"], lsa_ids, lin_ids)))
print("LSA и линейные источники готовы.")


In [ ]:
# Считаем BGE-M3 векторы корпуса на двух GPU с сохранением порядка.
"""BGE-M3 corpus encoding on two Kaggle GPUs, with order-safe checkpoints."""

import hashlib
from concurrent.futures import ThreadPoolExecutor

assert DEVICE.type == "cuda", "Включите GPU в Kaggle"
try:
    from sentence_transformers import SentenceTransformer
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers>=3,<6"])
    from sentence_transformers import SentenceTransformer


def discover_bge_source():
    if BGE_MODEL_SOURCE:
        return str(BGE_MODEL_SOURCE)
    local = [p.parent for p in Path("/kaggle/input").rglob("modules.json")
             if "bge" in str(p).lower() and "m3" in str(p).lower()]
    return str(local[0]) if local else "BAAI/bge-m3"


def raw_text(value):
    return "" if value is None or pd.isna(value) else str(value)


def bge_item_text(pos):
    return ("Заголовок: " + raw_text(items.item_title_raw.iat[pos])[:240] +
            "\nПараметры: " + raw_text(items.item_infm_params_text.iat[pos])[:250] +
            "\nОписание: " + raw_text(items.item_description_raw.iat[pos])[:900])


def bge_query_text(query):
    q = raw_text(query.get("search_query"))
    f = raw_text(query.get("search_infm_params_text"))
    return q + (". Фильтры: " + f if f else "")


BGE_SOURCE = discover_bge_source()
gpu_count = torch.cuda.device_count()
print("BGE-M3:", BGE_SOURCE, "| доступно GPU:", gpu_count)
primary_model = SentenceTransformer(BGE_SOURCE, device="cuda:0")
primary_model.max_seq_length = BGE_MAX_TOKENS
primary_model.half().eval()

# Отпечаток последовательности item_id защищает кеш от иной сортировки
# корпуса. Каждый результат записывается по исходному номеру строки.
id_digest = hashlib.sha256()
for item_id in items.item_id:
    id_digest.update(str(item_id).encode("ascii"))
    id_digest.update(b"\n")
fingerprint = {
    "n_items": N_ITEMS,
    "ordered_item_ids_sha256": id_digest.hexdigest(),
    "items_bytes": (DATA / "benchmark_items.parquet").stat().st_size,
    "max_tokens": BGE_MAX_TOKENS,
    "source": BGE_SOURCE,
}
bge_file = WORK / f"bge_m3_items_len{BGE_MAX_TOKENS}_ordered.npy"
bge_meta = WORK / f"bge_m3_items_len{BGE_MAX_TOKENS}_ordered.json"
saved_meta = json.loads(bge_meta.read_text()) if bge_meta.exists() else None
if bge_file.exists() and saved_meta and saved_meta.get("fingerprint") == fingerprint:
    bge_items = np.load(bge_file, mmap_mode="r+")
    done = int(saved_meta["done"])
    assert bge_items.shape == (N_ITEMS, 1024) and 0 <= done <= N_ITEMS
    print("Продолжаем с объявления", done)
else:
    probe = primary_model.encode([bge_item_text(0)], batch_size=1,
                                 normalize_embeddings=True,
                                 convert_to_numpy=True)
    assert probe.shape == (1, 1024), probe.shape
    bge_items = np.lib.format.open_memmap(
        bge_file, mode="w+", dtype=np.float16, shape=(N_ITEMS, 1024))
    done = 0


def encode_range(model, start, end):
    """The worker owns one model and one GPU; writing happens in the main thread."""
    texts = [bge_item_text(pos) for pos in range(start, end)]
    vectors = model.encode(texts, batch_size=BGE_BATCH,
                           normalize_embeddings=True,
                           convert_to_numpy=True, show_progress_bar=False)
    assert vectors.shape == (end - start, 1024)
    return start, end, vectors.astype(np.float16)


second_model = None
if done < N_ITEMS and gpu_count >= 2:
    second_model = SentenceTransformer(BGE_SOURCE, device="cuda:1")
    second_model.max_seq_length = BGE_MAX_TOKENS
    second_model.half().eval()

# Чекпоинт хранит только непрерывный префикс: при сбое пары чанков
# возобновление безопасно и исходный порядок не меняется.
if second_model is not None:
    with ThreadPoolExecutor(max_workers=2) as workers:
        for start in range(done, N_ITEMS, 2 * BGE_CHUNK):
            middle = min(start + BGE_CHUNK, N_ITEMS)
            end = min(start + 2 * BGE_CHUNK, N_ITEMS)
            pending = [workers.submit(encode_range, primary_model, start, middle)]
            if middle < end:
                pending.append(workers.submit(encode_range, second_model, middle, end))
            completed = [future.result() for future in pending]
            for lo, hi, vectors in completed:
                bge_items[lo:hi] = vectors
            bge_items.flush()
            bge_meta.write_text(json.dumps({"fingerprint": fingerprint, "done": end}))
            if end % 5000 < 2 * BGE_CHUNK or end == N_ITEMS:
                print("BGE embeddings (2 GPU):", end, "/", N_ITEMS, flush=True)
else:
    for start in range(done, N_ITEMS, BGE_CHUNK):
        end = min(start + BGE_CHUNK, N_ITEMS)
        lo, hi, vectors = encode_range(primary_model, start, end)
        bge_items[lo:hi] = vectors
        bge_items.flush()
        bge_meta.write_text(json.dumps({"fingerprint": fingerprint, "done": end}))
        if end % 5000 < BGE_CHUNK or end == N_ITEMS:
            print("BGE embeddings (1 GPU):", end, "/", N_ITEMS, flush=True)

assert json.loads(bge_meta.read_text())["done"] == N_ITEMS
assert bge_items.shape == (N_ITEMS, 1024)
# Несколько повторных кодирований проверяют порядок строк и целостность кеша.
sample_positions = np.unique(np.linspace(0, N_ITEMS - 1, num=7, dtype=np.int32))
sample_vectors = primary_model.encode(
    [bge_item_text(int(pos)) for pos in sample_positions],
    batch_size=BGE_BATCH, normalize_embeddings=True,
    convert_to_numpy=True)
alignment = np.sum(sample_vectors * np.asarray(bge_items[sample_positions], np.float32), axis=1)
assert np.all(alignment > 0.995), (sample_positions, alignment)
print("Проверка порядка BGE, min cosine:", round(float(alignment.min()), 5))

bge_queries = primary_model.encode(
    [bge_query_text(c["query"]) for c in cases],
    batch_size=BGE_BATCH, normalize_embeddings=True,
    convert_to_numpy=True, show_progress_bar=True).astype(np.float32)
del primary_model, second_model
gc.collect()
torch.cuda.empty_cache()
print("BGE-M3 vectors:", bge_items.shape, bge_queries.shape)


In [ ]:
# Добавляем BGE как семантический источник и признак последующих моделей.
bge_sources = retrieve_vector_sources(bge_items, bge_queries, "BGE source")
for case, source in zip(cases, bge_sources):
    case["bge_source"] = source
    case["pool_all"] = np.unique(np.concatenate((case["pool_both"], source)))
print("BGE-источник и полный пул готовы.")


In [ ]:
# Оцениваем подкатегории как мягкий приоритет, без жёсткого исключения.
prior_labels = sorted(label_counts)
prior_total = sum(label_counts.values())
def microcat_probabilities(query):
    present = [t for t in prior_terms(query.get("search_query"),
                                       query.get("search_infm_params_text"))
               if t in term_label_counts]
    if not present: return {}
    logp = {}
    for label in prior_labels:
        count = label_counts[label]
        value = math.log((count+1)/(prior_total+len(prior_labels)))
        for term in present:
            value += math.log((term_label_counts[term][label]+0.5)/(count+1))
        logp[label] = value
    top = max(logp.values())
    probabilities = {label: math.exp((value-top)/1.5) for label, value in logp.items()}
    z = sum(probabilities.values())
    return {label: value/z for label, value in probabilities.items()}

def inverse_rank_bonus(ids, source, strength, scale):
    rank = {int(pos): r for r, pos in enumerate(source, 1)}
    return np.fromiter((strength / (1+r/scale) if (r := rank.get(int(pos))) else 0.0
                        for pos in ids), dtype=np.float32, count=len(ids))

def cheap_scores(case, ids):
    query = case["query"]
    # Для BM25 используем прежний 15/sqrt(rank); LSA и linear добавляют
    # независимые мягкие сигналы, не исключающие нелокальные объявления.
    lexical_rank = {int(pos): r for r, pos in enumerate(case["lex"], 1)}
    score = np.fromiter((15/math.sqrt(r) if (r := lexical_rank.get(int(pos))) else 0.0
                         for pos in ids), dtype=np.float32, count=len(ids))
    score += 5.0 * (locs[ids] == as_int(query.get("search_location_id")))
    prior = microcat_probabilities(query)
    score += 3.0 * np.fromiter((prior.get(int(m), 0.0) for m in microcats[ids]),
                               dtype=np.float32, count=len(ids))
    score += 4.0 * inverse_rank_bonus(ids, case["lsa"], 1.0, 100.0)
    score += 4.0 * inverse_rank_bonus(ids, case["linear"], 1.0, 100.0)
    score += 0.12 * np.log1p(reviews[ids])
    score += np.fromiter((0.5/math.sqrt(r) if (r := case["prefixes"].get(int(pos))) else 0.0
                          for pos in ids), dtype=np.float32, count=len(ids))
    return score

# BGE-векторы копируются на GPU один раз; для каждого запроса оценивается
# максимальный pool_all, а меньшие варианты берут его подмножество.
bge_iv = torch.as_tensor(np.asarray(bge_items), device=DEVICE, dtype=torch.float16)
for n, case in enumerate(cases, 1):
    ids = case["pool_all"]
    case["cheap_all"] = cheap_scores(case, ids)
    with torch.no_grad():
        q = torch.as_tensor(bge_queries[n-1], device=DEVICE, dtype=torch.float32)
        gathered = bge_iv[torch.as_tensor(ids.astype(np.int64), device=DEVICE)]
        case["bge_all"] = (gathered.float() @ q).cpu().numpy()
    if n % 250 == 0: print("BGE pool scores:", n, "/", len(cases), flush=True)
del bge_iv
torch.cuda.empty_cache()

POOL_FIELDS = ("base", "pool_lsa", "pool_both", "pool_all")
def ranks_desc(values):
    order = np.argsort(-values, kind="stable")
    rank = np.empty(len(order), dtype=np.int32)
    rank[order] = np.arange(1, len(order)+1, dtype=np.int32)
    return rank

# Предвычисляем ранги для каждой версии пула; такое подмножество держит
# сравнение BGE и дешёвого отбора строго на одинаковом наборе объявлений.
for case in cases:
    for field in POOL_FIELDS:
        ids = case[field]
        offsets = np.searchsorted(case["pool_all"], ids)
        assert np.array_equal(case["pool_all"][offsets], ids)
        case[field+"_rank"] = (ranks_desc(case["cheap_all"][offsets]),
                               ranks_desc(case["bge_all"][offsets]))
print("Дешёвые и BGE-баллы внутри широких пулов готовы.")


In [ ]:
# Подготавливаем общий пул и дешёвые баллы для L1.
STAGE2_POOL = "pool_both"
STAGE2_ALPHA = 0.25  # Значение зафиксировано до нового holdout.
for case in cases:
    ids = case[STAGE2_POOL]
    cheap_rank, bge_rank = case[STAGE2_POOL + "_rank"]
    fusion = ((1 - STAGE2_ALPHA) / (60.0 + cheap_rank)
              + STAGE2_ALPHA / (60.0 + bge_rank))
    pool_set = set(map(int, ids))
    orders = {
        "cheap": ids[np.argsort(cheap_rank, kind="stable")],
        "bge": ids[np.argsort(bge_rank, kind="stable")],
        "fusion": ids[np.argsort(-fusion, kind="stable")],
    }
    for source in ("lex", "lsa", "linear"):
        orders[source] = np.fromiter(
            (int(pos) for pos in case[source] if int(pos) in pool_set),
            dtype=np.int32)
    case["_s2_orders"] = orders
print("Источники L1 подготовлены.")


## 3. CatBoost L1 и L2

L1 быстро оставляет 1500 объявлений. L2 считает более подробные признаки пары и оставляет 750. При обучении используются группы запросов и OOF-предсказания для следующих этапов.


In [ ]:
# Признаки и обучение CatBoost L1 для широкого пула.
# Обучаем на новом development-разбиении: они НЕ входили в обучение linear.
# Это честнее, чем обучать CatBoost на тех train-парах, которые уже запомнил linear.
try:
    from catboost import CatBoostRanker, Pool
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "catboost>=1.2,<2"])
    from catboost import CatBoostRanker, Pool

assert DEVICE.type == "cuda", "Включите GPU в Kaggle"
category = pd.to_numeric(items.item_category_id, errors="coerce").fillna(-1).to_numpy(np.int32)
rating = pd.to_numeric(items.item_rating, errors="coerce").fillna(-1).to_numpy(np.float32)
price = np.log1p(pd.to_numeric(items.item_price, errors="coerce").fillna(0)
                 .clip(lower=0).to_numpy(np.float32))
lat = pd.to_numeric(items.item_latitude, errors="coerce").to_numpy(np.float32)
lon = pd.to_numeric(items.item_longitude, errors="coerce").to_numpy(np.float32)
contact_flags = pq.read_table(DATA / "benchmark_items.parquet", columns=[
    "item_is_phone_hidden", "item_is_message_forbidden"]).to_pydict()
phone_hidden = np.asarray([int(x or 0) for x in contact_flags["item_is_phone_hidden"]], np.float32)
message_forbidden = np.asarray([int(x or 0) for x in contact_flags["item_is_message_forbidden"]], np.float32)
title_terms = [set(tokens(t, 240)) for t in items.item_title_raw]
param_terms = [set(tokens(p, 800)) for p in items.item_infm_params_text]

FEATURE_NAMES = [
    "bm25_score", "log_bm25_rank", "lsa_cos", "log_lsa_rank",
    "linear_cos", "log_linear_rank", "bge_cos", "log_bge_rank",
    "cheap_score", "log_cheap_rank", "same_location", "log_distance_km",
    "category_match", "microcat_prior", "log_reviews", "rating", "log_price",
    "phone_hidden", "message_forbidden", "title_overlap", "params_overlap",
    "log_prefix_rank", "query_terms",
    "has_filter", "delivery_search", "log_pool_size",
]

# Корпусные векторы уже посчитаны предыдущими ячейками. Держим их на GPU.
lsa_gpu = torch.as_tensor(np.asarray(lsa_items), device=DEVICE, dtype=torch.float16)
linear_gpu = torch.as_tensor(np.asarray(linear_items), device=DEVICE, dtype=torch.float16)

def feature_matrix(case_index, ids):
    case = cases[case_index]
    ids = np.asarray(ids, dtype=np.int32)
    query = case["query"]
    n = len(ids)
    # Для каждого запроса feature_matrix вызывается один раз; не храним
    # тысячи больших словарей BM25 в памяти после обработки запроса.
    bm25 = lexical_hits(query)
    ranks = {
        source: {int(pos): r for r, pos in enumerate(case[source], 1)}
        for source in ("lex", "lsa", "linear")
    }
    def source_rank(source):
        return np.fromiter((ranks[source].get(int(pos), 20_000) for pos in ids),
                           dtype=np.float32, count=n)

    both_offset = np.searchsorted(case[STAGE2_POOL], ids)
    all_offset = np.searchsorted(case["pool_all"], ids)
    assert np.array_equal(case[STAGE2_POOL][both_offset], ids)
    assert np.array_equal(case["pool_all"][all_offset], ids)
    with torch.no_grad():
        idx_gpu = torch.as_tensor(ids.astype(np.int64), device=DEVICE)
        lsa_cos = (lsa_gpu[idx_gpu].float() @ torch.as_tensor(
            lsa_queries[case_index], device=DEVICE)).cpu().numpy()
        linear_cos = (linear_gpu[idx_gpu].float() @ torch.as_tensor(
            linear_queries[case_index], device=DEVICE)).cpu().numpy()

    center = centers.get(as_int(query.get("search_location_id")))
    if center is None:
        distance = np.full(n, 10_000.0, np.float32)
    else:
        a1, a2 = np.radians(center[0]), np.radians(lat[ids])
        delta_lat = a2 - a1
        delta_lon = np.radians(lon[ids] - center[1])
        h = np.sin(delta_lat/2)**2 + np.cos(a1)*np.cos(a2)*np.sin(delta_lon/2)**2
        distance = (12742.0 * np.arcsin(np.sqrt(np.clip(h, 0, 1)))).astype(np.float32)
        distance = np.nan_to_num(distance, nan=10_000.0)

    q_terms = set(tokens(query.get("search_query")))
    f_terms = set(tokens(query.get("search_infm_params_text"))) - GENERIC_FILTER
    prior = microcat_probabilities(query)  # dev/holdout исключены из обучения prior.
    title_overlap = np.fromiter(
        (len(q_terms & title_terms[int(pos)]) / max(1, len(q_terms)) for pos in ids),
        dtype=np.float32, count=n)
    params_overlap = np.fromiter(
        (len(f_terms & param_terms[int(pos)]) / max(1, len(f_terms)) for pos in ids),
        dtype=np.float32, count=n)
    prefix_rank = np.fromiter(
        (case["prefixes"].get(int(pos), 20_000) for pos in ids),
        dtype=np.float32, count=n)
    prior_value = np.fromiter(
        (prior.get(int(microcats[pos]), 0.0) for pos in ids),
        dtype=np.float32, count=n)

    x = np.column_stack([
        np.fromiter((bm25.get(int(pos), 0.0) for pos in ids), np.float32, n),
        np.log1p(source_rank("lex")), lsa_cos, np.log1p(source_rank("lsa")),
        linear_cos, np.log1p(source_rank("linear")),
        case["bge_all"][all_offset],
        np.log1p(case[STAGE2_POOL + "_rank"][1][both_offset]),
        case["cheap_all"][all_offset],
        np.log1p(case[STAGE2_POOL + "_rank"][0][both_offset]),
        (locs[ids] == as_int(query.get("search_location_id"))).astype(np.float32),
        np.log1p(distance),
        (category[ids] == as_int(query.get("search_category"))).astype(np.float32),
        prior_value, np.log1p(reviews[ids]), rating[ids], price[ids],
        phone_hidden[ids], message_forbidden[ids],
        title_overlap, params_overlap, np.log1p(prefix_rank),
        np.full(n, len(q_terms), np.float32),
        np.full(n, bool(f_terms), np.float32),
        np.full(n, as_int(query.get("search_is_delivery_search"), 0), np.float32),
        np.full(n, np.log1p(len(case[STAGE2_POOL])), np.float32),
    ]).astype(np.float32)
    assert x.shape == (n, len(FEATURE_NAMES))
    return x

# Не размечаем кандидата отрицательным, если он был выбран хотя бы в одном
# событии с тем же исходным текстом запроса. Остальные 0 — слабые метки.
known_by_text = {}
for case in cases[:N_DEV]:
    text = case["query"]["search_query"]
    known_by_text[text] = set().union(*(events[key] for key in by_text[text]))

sample_rng = np.random.default_rng(SEED)
train_x, train_y, train_group = [], [], []
for case_index in range(N_DEV):
    case = cases[case_index]
    pool_ids = case[STAGE2_POOL]
    order = case["_s2_orders"]
    boundary = order["cheap"][250:min(1500, len(pool_ids))]
    candidate_neg = list(order["cheap"][:35]) + list(order["bge"][:35])
    candidate_neg += list(order["lex"][:30]) + list(order["linear"][:40])
    candidate_neg += list(order["lsa"][:20])
    if len(boundary):
        candidate_neg += list(sample_rng.choice(boundary, min(70, len(boundary)), replace=False))
    candidate_neg += list(sample_rng.choice(pool_ids, min(45, len(pool_ids)), replace=False))
    forbidden = known_by_text[case["query"]["search_query"]]
    negatives = [int(pos) for pos in dict.fromkeys(candidate_neg) if int(pos) not in forbidden]
    positives = sorted(set(case["gold"]) & set(map(int, pool_ids)))
    if not positives or not negatives:
        continue
    selected_ids = np.asarray(positives + negatives, dtype=np.int32)
    train_x.append(feature_matrix(case_index, selected_ids))
    train_y.append(np.r_[np.ones(len(positives)), np.zeros(len(negatives))])
    train_group.append(np.full(len(selected_ids), case_index, dtype=np.int32))
    if (case_index + 1) % 100 == 0:
        print("Training groups:", case_index + 1, "/", N_DEV, flush=True)

train_pool = Pool(np.vstack(train_x), label=np.concatenate(train_y),
                  group_id=np.concatenate(train_group), feature_names=FEATURE_NAMES)
print("CatBoost training rows:", train_pool.num_row(), "groups:", len(train_x))
stage2_ranker = CatBoostRanker(iterations=500, depth=5, learning_rate=0.04,
                               l2_leaf_reg=8, loss_function="QuerySoftMax",
                               task_type="GPU", devices="0", random_seed=SEED,
                               verbose=100)
stage2_ranker.fit(train_pool)
stage2_ranker.save_model(str(WORK / "stage2_catboost_querysoftmax.cbm"))

# Оценка идёт по ПОЛНОМУ широкому пулу, а не по обучающей выборке негативов.
for case_index in range(N_DEV, len(cases)):
    case = cases[case_index]
    ids = case[STAGE2_POOL]
    if not len(ids):
        case["_cat_order"] = ids
        continue
    score = stage2_ranker.predict(feature_matrix(case_index, ids))
    case["_cat_order"] = ids[np.argsort(-score, kind="stable")]
    if (case_index + 1) % 100 == 0:
        print("Full-pool predictions:", case_index + 1, "/", len(cases), flush=True)


In [ ]:
# OOF-ранги L1 не дают L2 учиться на предсказаниях того же fold.
# Пять моделей L1 для честных рангов обучающих запросов.
assert len(train_x) == len(train_y) == len(train_group)
fold_ids = np.empty(N_DEV, dtype=np.int8)
fold_ids[np.random.default_rng(SEED + 2).permutation(N_DEV)] = (
    np.arange(N_DEV) % 3)
train_case_ids = np.asarray([int(group[0]) for group in train_group])
for fold in range(3):
    tr = np.flatnonzero(fold_ids[train_case_ids] != fold)
    va = np.flatnonzero(fold_ids == fold)
    fold_pool = Pool(
        np.vstack([train_x[j] for j in tr]),
        label=np.concatenate([train_y[j] for j in tr]),
        group_id=np.concatenate([train_group[j] for j in tr]),
        feature_names=FEATURE_NAMES,
    )
    fold_ranker = CatBoostRanker(
        iterations=500, depth=5, learning_rate=0.04, l2_leaf_reg=8,
        loss_function="QuerySoftMax", task_type="GPU", devices="0",
        random_seed=SEED + fold + 10, verbose=False,
    )
    fold_ranker.fit(fold_pool)
    for i in va:
        ids = cases[int(i)][STAGE2_POOL]
        if not len(ids):
            cases[int(i)]["_cat_oof_order"] = ids
            continue
        score = fold_ranker.predict(feature_matrix(int(i), ids))
        cases[int(i)]["_cat_oof_order"] = ids[np.argsort(-score, kind="stable")]
    print(f"OOF L1: {fold+1}/3; обучающих групп {len(tr)}", flush=True)
    del fold_ranker, fold_pool
    gc.collect()
assert all("_cat_oof_order" in c for c in cases[:N_DEV])
assert all("_cat_order" in c for c in cases[N_DEV:])


In [ ]:
# Дополнительные текстовые признаки пары для второго CatBoost.
from functools import lru_cache

FINAL_K = 1500  # Зафиксирован до этого опыта.
FINAL_FEATURE_NAMES = FEATURE_NAMES + [
    "log_l1_rank", "tfidf_cos", "title_exact_phrase", "title_all_query_terms",
    "description_overlap", "description_all_query_terms",
    "combined_text_overlap", "log_title_length", "log_description_length",
    "all_filter_terms_in_params",
]
normalized_titles = [norm(title, 240) for title in items.item_title_raw]

@lru_cache(maxsize=120_000)
def description_words(pos):
    return frozenset(tokens(items.item_description_raw.iat[int(pos)], 1600))

def final_features(case_index, ids, l1_ranks):
    ids = np.asarray(ids, dtype=np.int32)
    l1_ranks = np.asarray(l1_ranks, dtype=np.float32)
    assert ids.shape == l1_ranks.shape
    case = cases[case_index]
    query = case["query"]
    words = set(tokens(query.get("search_query")))
    filters = set(tokens(query.get("search_infm_params_text"))) - GENERIC_FILTER
    phrase = norm(query.get("search_query"))
    denominator = max(1, len(words))

    # Общая TF–IDF основа уже построена; это точный разреженный cosine.
    tfidf_cos = (X_items[ids] @ X_val_q[case_index].T).toarray().ravel()
    title_phrase = np.fromiter(
        (bool(phrase) and phrase in normalized_titles[int(pos)] for pos in ids),
        dtype=np.float32, count=len(ids))
    title_all = np.fromiter(
        (bool(words) and words <= title_terms[int(pos)] for pos in ids),
        dtype=np.float32, count=len(ids))
    desc_overlap = np.fromiter(
        (len(words & description_words(pos)) / denominator for pos in ids),
        dtype=np.float32, count=len(ids))
    desc_all = np.fromiter(
        (bool(words) and words <= description_words(pos) for pos in ids),
        dtype=np.float32, count=len(ids))
    combined_overlap = np.fromiter(
        (len(words & (title_terms[int(pos)] | param_terms[int(pos)] |
                      description_words(pos))) / denominator for pos in ids),
        dtype=np.float32, count=len(ids))
    title_length = np.fromiter(
        (len(normalized_titles[int(pos)]) for pos in ids),
        dtype=np.float32, count=len(ids))
    desc_length = np.fromiter(
        (len(description_words(pos)) for pos in ids),
        dtype=np.float32, count=len(ids))
    filter_all = np.fromiter(
        (bool(filters) and filters <= param_terms[int(pos)] for pos in ids),
        dtype=np.float32, count=len(ids))

    extra = np.column_stack([
        np.log1p(l1_ranks), tfidf_cos, title_phrase, title_all, desc_overlap, desc_all,
        combined_overlap, np.log1p(title_length), np.log1p(desc_length),
        filter_all,
    ]).astype(np.float32)
    result = np.hstack((feature_matrix(case_index, ids), extra))
    assert result.shape == (len(ids), len(FINAL_FEATURE_NAMES))
    return result


In [ ]:
# Подбираем обучающие кандидаты и слабые негативы для L2.
# Для всех групп одинаковые кандидаты и негативы. Неизвестные
# положительные объявления того же текста не превращаем в негативы.
final_rng = np.random.default_rng(SEED + 1)
final_blocks = []
for case_index in range(N_DEV):
    case = cases[case_index]
    order = case["_cat_oof_order"][:FINAL_K]
    positive_at = np.flatnonzero(np.isin(order, list(case["gold"])))
    if not len(positive_at):
        continue
    text = case["query"]["search_query"]
    forbidden = set().union(*(events[key] for key in by_text[text]))
    negative_at = []
    for lo, hi, quota in ((0, 100, 100), (100, 500, 80),
                          (500, 1000, 60), (1000, FINAL_K, 60)):
        available = np.arange(lo, min(hi, len(order)))
        available = np.asarray([r for r in available
                                if int(order[r]) not in forbidden], dtype=np.int32)
        if len(available):
            negative_at.extend(final_rng.choice(
                available, min(quota, len(available)), replace=False))
    if not negative_at:
        continue
    selected_at = np.r_[positive_at, negative_at].astype(np.int32)
    x = final_features(case_index, order[selected_at], selected_at + 1)
    y = np.r_[np.ones(len(positive_at)), np.zeros(len(negative_at))]
    group = np.full(len(selected_at), case_index, dtype=np.int32)
    final_blocks.append((x, y, group))
    if (case_index + 1) % 250 == 0:
        print("Финальные обучающие группы:", case_index + 1, "/", N_DEV,
              flush=True)

assert len(final_blocks) > 500
print("Пригодных групп:", len(final_blocks),
      "| обучение L2:", len(final_blocks))


In [ ]:
# Обучаем L2 и считаем его OOF top-750.
BRIDGE_K = 750
L2_NAMES = FINAL_FEATURE_NAMES
L3_BASE_NAMES = FINAL_FEATURE_NAMES + ["log_l2_rank"]

def ranker_for_stage(seed, iterations=600, verbose=False):
    return CatBoostRanker(
        iterations=iterations, depth=6, learning_rate=0.04,
        l2_leaf_reg=10, loss_function="QuerySoftMax",
        task_type="GPU", devices="0", random_seed=seed, verbose=verbose)

l2_pool = Pool(
    np.vstack([part[0] for part in final_blocks]),
    label=np.concatenate([part[1] for part in final_blocks]),
    group_id=np.concatenate([part[2] for part in final_blocks]),
    feature_names=L2_NAMES,
)
l2_ranker = ranker_for_stage(SEED + 1, verbose=100)
l2_ranker.fit(l2_pool)
l2_ranker.save_model(str(WORK / "stage1500_to750.cbm"))
print("L2 обучен:", len(final_blocks), "групп,", l2_pool.num_row(), "строк")
del l2_pool
gc.collect()

# Кеш признаков нужен для OOF L2 и для одинаковых обучающих пар обоих L3.
l2_dev_features = {}
for i in range(N_DEV):
    ids = cases[i]["_cat_oof_order"][:FINAL_K]
    x = (final_features(i, ids, np.arange(1, len(ids) + 1))
         if len(ids) else np.empty((0, len(L2_NAMES)), np.float32))
    l2_dev_features[i] = (ids, x)
    if (i + 1) % 250 == 0:
        print("Полные признаки L2:", i + 1, "/", N_DEV, flush=True)

l2_fold_ids = np.empty(N_DEV, np.int8)
l2_fold_ids[np.random.default_rng(SEED + 51).permutation(N_DEV)] = (
    np.arange(N_DEV) % 3)
for fold in range(3):
    training = [part for part in final_blocks
                if l2_fold_ids[int(part[2][0])] != fold]
    fold_pool = Pool(
        np.vstack([part[0] for part in training]),
        label=np.concatenate([part[1] for part in training]),
        group_id=np.concatenate([part[2] for part in training]),
        feature_names=L2_NAMES,
    )
    fold_ranker = ranker_for_stage(SEED + 61 + fold)
    fold_ranker.fit(fold_pool)
    for i in np.flatnonzero(l2_fold_ids == fold):
        i = int(i)
        ids, x = l2_dev_features[i]
        if not len(ids):
            cases[i]["_l2_oof_order"] = ids
            continue
        scores = fold_ranker.predict(x)
        cases[i]["_l2_oof_order"] = ids[np.argsort(-scores, kind="stable")]
    print("OOF L2:", fold + 1, "/3", flush=True)
    del fold_ranker, fold_pool, training
    gc.collect()
assert all("_l2_oof_order" in c for c in cases[:N_DEV])

# Это только development-оценка потолка нового промежуточного этапа.
dev_l2_ceiling = float(np.mean([
    len(c["gold"].intersection(c["_l2_oof_order"][:BRIDGE_K])) / len(c["gold"])
    for c in cases[:N_DEV]
]))
print("Development OOF Recall@750:", round(dev_l2_ceiling, 4))


## 4. LambdaMART для финального отбора

На том же пуле top-750 строим признаки и обучаем XGBoost LambdaMART с акцентом на верх списка. Его балл служит основой финального ранжирования.


In [ ]:
# Собираем уже готовые OOF-признаки для обучения L3.
# Строим guard_full из уже готовых OOF L1/L2 без графовых признаков.
guard_full = {}
for i in range(N_DEV):
    l1_ids, l2_x = l2_dev_features[i]
    bridge = np.asarray(cases[i]["_l2_oof_order"][:BRIDGE_K], np.int32)
    if len(bridge):
        position = {int(pos): j for j, pos in enumerate(l1_ids)}
        at = np.asarray([position[int(pos)] for pos in bridge], np.int32)
        base_x = np.column_stack((
            l2_x[at], np.log1p(np.arange(1, len(bridge)+1)))).astype(np.float32)
    else:
        base_x = np.empty((0, len(L3_BASE_NAMES)), np.float32)
    guard_full[i] = (bridge, base_x, None)
assert len(guard_full) == N_DEV
print("OOF top-750 готов для", len(guard_full), "development-запросов")


In [ ]:
# Расширяем обучение L3 до 10 тысяч групп и обучаем LambdaMART-topk.
# Ячейка 2. Кривая обучения L3 при неизменном проверочном top-750.
# Переиспользуются корпусные TF-IDF/LSA/linear/BGE-векторы и модели L1/L2.
# Для дополнительных запросов строится тот же широкий пул и тот же каскад.
# Мы намеренно сравниваем на общем наборе БАЗОВЫХ L3-признаков: граф/история
# содержат выборы train и на новых обучающих запросах дали бы утечку ответа.
# Новые группы до 10000; если пригодных меньше, выводится фактическое число.
import gc, hashlib, joblib, time
from collections import defaultdict
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
from catboost import CatBoostRanker, Pool

required = ["cases", "events", "by_text", "excluded_texts", "make_broad_pool",
            "retrieve_vector_sources", "cheap_scores", "ranks_desc", "feature_matrix",
            "final_features", "stage2_ranker", "l2_ranker", "guard_full",
            "L3_BASE_NAMES", "BGE_SOURCE", "bge_items", "bge_query_text"]
assert all(name in globals() for name in required), "Сначала выполните основной ноутбук"
assert len(cases) == N_DEV + N_HOLDOUT
assert len(guard_full) >= N_DEV - 30
assert STAGE2_POOL == "pool_both" and FINAL_K == 1500 and BRIDGE_K == 750

_orig_cases = cases
_orig_X_val_q = X_val_q
_orig_lsa_queries = lsa_queries
_orig_linear_queries = linear_queries
_orig_bge_queries = bge_queries
_l3_t0 = time.perf_counter()

# В прошлом опыте переменная tfidf случайно затиралась массивом. Восстановим
# именно векторизатор, не меняя уже готовых матриц и модели.
if not hasattr(tfidf, "vocabulary_"):
    tfidf = joblib.load(tfidf_path)
assert hasattr(tfidf, "transform")

def _sample_l3_group(ids, x, gold, forbidden, rng):
    """Все positive в top-750 плюс одинаковая стратификация слабых negative."""
    pos_at = np.flatnonzero(np.isin(ids, list(gold)))
    if not len(pos_at):
        return None
    neg_at = []
    for lo, hi, quota in ((0, 50, 50), (50, 150, 75),
                          (150, 350, 75), (350, 750, 100)):
        available = np.asarray([
            r for r in range(lo, min(hi, len(ids)))
            if int(ids[r]) not in forbidden
        ], np.int32)
        if len(available):
            neg_at.extend(rng.choice(available, min(quota, len(available)),
                                     replace=False).tolist())
    if not neg_at:
        return None
    at = np.r_[pos_at, neg_at].astype(np.int32)
    return (np.asarray(x[at, :len(L3_BASE_NAMES)], np.float32),
            np.r_[np.ones(len(pos_at)), np.zeros(len(neg_at))].astype(np.float32))

_rng = np.random.default_rng(SEED + 2741)
expanded_l3_blocks = []
for i in range(N_DEV):
    if i not in guard_full:
        continue
    bridge, base_x, _ = guard_full[i]
    text = cases[i]["query"]["search_query"]
    forbidden = set().union(*(events[key] for key in by_text[text]))
    part = _sample_l3_group(bridge, base_x, cases[i]["gold"], forbidden, _rng)
    if part is not None:
        expanded_l3_blocks.append(part)
original_groups = len(expanded_l3_blocks)
print("Базовых development-групп:", original_groups)

# Дополнительные тексты берём из train только при выбранном item_id в корпусе.
# Тексты двух текущих проверочных групп исключены ДО построения примеров.
extra_texts = [text for text in sorted(by_text, key=str)
               if text not in excluded_texts]
np.random.default_rng(SEED + 2742).shuffle(extra_texts)
extra_texts = extra_texts[:10_800]  # запас на потери в wide/L1/L2
extra_cases = []
for text in extra_texts:
    keys = [key for key in by_text[text] if key[-1] == 114] or by_text[text]
    key = max(keys, key=lambda event: (len(events[event]), str(event)))
    extra_cases.append({"query": dict(zip(Q_COLS, key)), "gold": events[key]})
print("Новых текстов для подготовки:", len(extra_cases))

# Кодируем только НОВЫЕ запросы; item-эмбеддинги BGE из предыдущей ячейки
# уже рассчитаны двумя GPU и остаются в исходном порядке benchmark_items.
new_X = tfidf.transform([query_text(c["query"]) for c in extra_cases]).tocsr()
new_lsa = l2_normalize(lsa.transform(new_X).astype(np.float32))
_linear_probe = SparseLinearTower(X_items.shape[1], LINEAR_DIM).to(
    device=DEVICE, dtype=torch.float32)
_linear_probe.load_state_dict(torch.load(
    WORK / "linear_towers.pt", map_location=DEVICE, weights_only=True))
_linear_probe.eval()
new_linear = np.empty((len(extra_cases), LINEAR_DIM), np.float32)
with torch.no_grad():
    for lo in range(0, len(extra_cases), 2048):
        hi = min(lo + 2048, len(extra_cases))
        new_linear[lo:hi] = _linear_probe.encode(new_X[lo:hi], "query").cpu().numpy()
del _linear_probe
torch.cuda.empty_cache()

from sentence_transformers import SentenceTransformer
_bge_query_device = "cuda:1" if torch.cuda.device_count() > 1 else "cuda:0"
_bge_query_model = SentenceTransformer(BGE_SOURCE, device=_bge_query_device)
new_bge = _bge_query_model.encode(
    [bge_query_text(c["query"]) for c in extra_cases],
    batch_size=BGE_BATCH, normalize_embeddings=True,
    convert_to_numpy=True, show_progress_bar=True).astype(np.float32)
del _bge_query_model
gc.collect(); torch.cuda.empty_cache()

# Широкий пул и три источника строим с прежними функциями. Они читают
# глобальную переменную cases; временная подмена строго закрыта finally.
for n, c in enumerate(extra_cases, 1):
    base, lex, prefixes = make_broad_pool(c["query"])
    c.update(base=base, lex=lex, prefixes=prefixes)
    if n % 500 == 0:
        print("Широкие пулы:", n, "/", len(extra_cases), flush=True)
try:
    cases = extra_cases
    new_lsa_src = retrieve_vector_sources(lsa_items, new_lsa, "extra LSA")
    new_linear_src = retrieve_vector_sources(linear_items, new_linear, "extra linear")
    new_bge_src = retrieve_vector_sources(bge_items, new_bge, "extra BGE")
finally:
    cases = _orig_cases

for c, lsa_src, lin_src, bge_src in zip(
        extra_cases, new_lsa_src, new_linear_src, new_bge_src):
    c["lsa"], c["linear"], c["bge_source"] = lsa_src, lin_src, bge_src
    c["pool_both"] = np.unique(np.concatenate((c["base"], lsa_src, lin_src)))
    c["pool_all"] = np.unique(np.concatenate((c["pool_both"], bge_src)))
del new_lsa_src, new_linear_src, new_bge_src

# Для feature_matrix/final_features индексы новых строк идут после старых.
cases = _orig_cases + extra_cases
X_val_q = sp.vstack((_orig_X_val_q, new_X), format="csr")
lsa_queries = np.vstack((_orig_lsa_queries, new_lsa))
linear_queries = np.vstack((_orig_linear_queries, new_linear))
bge_queries = np.vstack((_orig_bge_queries, new_bge))

try:
    _bge_items_gpu = torch.as_tensor(
        np.asarray(bge_items), device=DEVICE, dtype=torch.float16)
    extra_wide_hits = extra_l1_hits = extra_l2_hits = 0
    for j, c in enumerate(extra_cases, 1):
        idx = len(_orig_cases) + j - 1
        ids = c["pool_all"]
        c["cheap_all"] = cheap_scores(c, ids)
        with torch.no_grad():
            ids_gpu = torch.as_tensor(ids.astype(np.int64), device=DEVICE)
            q_gpu = torch.as_tensor(new_bge[j-1], device=DEVICE)
            c["bge_all"] = (_bge_items_gpu[ids_gpu].float() @ q_gpu).cpu().numpy()
        both = c[STAGE2_POOL]
        off = np.searchsorted(ids, both)
        assert np.array_equal(ids[off], both)
        c[STAGE2_POOL + "_rank"] = (
            ranks_desc(c["cheap_all"][off]), ranks_desc(c["bge_all"][off]))

        gold = set(map(int, c["gold"]))
        extra_wide_hits += bool(gold & set(map(int, both)))
        if not len(both):
            continue
        l1_score = np.asarray(stage2_ranker.predict(feature_matrix(idx, both))).ravel()
        l1 = both[np.argsort(-l1_score, kind="stable")[:FINAL_K]]
        extra_l1_hits += bool(gold & set(map(int, l1)))
        x2 = final_features(idx, l1, np.arange(1, len(l1) + 1))
        l2_score = np.asarray(l2_ranker.predict(x2)).ravel()
        l2_at = np.argsort(-l2_score, kind="stable")[:BRIDGE_K]
        bridge = l1[l2_at]
        extra_l2_hits += bool(gold & set(map(int, bridge)))
        base_x = np.column_stack((x2[l2_at],
                                  np.log1p(np.arange(1, len(bridge) + 1))))
        forbidden = set().union(*(events[key] for key in by_text[c["query"]["search_query"]]))
        part = _sample_l3_group(bridge, base_x, gold, forbidden, _rng)
        if part is not None:
            expanded_l3_blocks.append(part)
        # Эти большие массивы не нужны следующему запросу.
        for name in ("base", "lex", "lsa", "linear", "bge_source",
                     "pool_both", "pool_all", "cheap_all", "bge_all",
                     STAGE2_POOL + "_rank", "prefixes"):
            c.pop(name, None)
        if j % 250 == 0:
            print("Дополнительные L3-группы:", j, "/", len(extra_cases),
                  "| пригодных:", len(expanded_l3_blocks)-original_groups,
                  "| минут:", round((time.perf_counter()-_l3_t0)/60, 1),
                  flush=True)
    del _bge_items_gpu
finally:
    # Не портим исходную Kaggle-сессию для третьей ячейки и последующих опытов.
    cases = _orig_cases
    X_val_q = _orig_X_val_q
    lsa_queries = _orig_lsa_queries
    linear_queries = _orig_linear_queries
    bge_queries = _orig_bge_queries
    torch.cuda.empty_cache()

print("Дополнительные потолки по запросам с ≥1 выбранным item:",
      {"wide": round(extra_wide_hits/max(1,len(extra_cases)), 4),
       "L1": round(extra_l1_hits/max(1,len(extra_cases)), 4),
       "L2": round(extra_l2_hits/max(1,len(extra_cases)), 4)})
print("Групп всего:", len(expanded_l3_blocks))

# Один и тот же проверочный top-750 и матрица признаков для всех моделей.
expanded_l3_hold = []
for i in range(N_DEV, N_DEV + N_HOLDOUT):
    c = cases[i]
    l1 = np.asarray(c["_cat_order"][:FINAL_K], np.int32)
    x2 = final_features(i, l1, np.arange(1, len(l1)+1))
    l2_score = np.asarray(l2_ranker.predict(x2)).ravel()
    at = np.argsort(-l2_score, kind="stable")[:BRIDGE_K]
    bridge = l1[at]
    x = np.column_stack((x2[at], np.log1p(np.arange(1, len(bridge)+1)))).astype(np.float32)
    expanded_l3_hold.append((bridge, x, set(map(int, c["gold"]))))

def _fit_expanded_l3(blocks, kind, seed):
    xs, ys, gids = [], [], []
    for g, (x, y) in enumerate(blocks):
        xs.append(x); ys.append(y); gids.append(np.full(len(y), g, np.int32))
    X = np.vstack(xs); y = np.concatenate(ys); gid = np.concatenate(gids)
    if kind == "QuerySoftMax":
        ranker = CatBoostRanker(
            iterations=600, depth=6, learning_rate=0.04, l2_leaf_reg=10,
            loss_function="QuerySoftMax", task_type="GPU", devices="0",
            random_seed=seed, verbose=False)
        ranker.fit(Pool(X, label=y, group_id=gid, feature_names=L3_BASE_NAMES))
    else:
        import xgboost as xgb
        kwargs = dict(objective="rank:ndcg", tree_method="hist", device="cuda",
                      n_estimators=500, max_depth=6, learning_rate=0.05,
                      subsample=0.9, colsample_bytree=0.9,
                      lambdarank_pair_method="topk",
                      lambdarank_num_pair_per_sample=60, random_state=seed)
        ranker = xgb.XGBRanker(**kwargs)
        try:
            ranker.fit(X, y, qid=gid, verbose=False)
        except Exception as exc:
            print("XGBoost GPU недоступен, пробуем CPU:", str(exc)[:180])
            ranker = xgb.XGBRanker(**{**kwargs, "device":"cpu"})
            ranker.fit(X, y, qid=gid, verbose=False)
    return ranker

def _score_expanded_l3(ranker):
    values = []
    for bridge, x, gold in expanded_l3_hold:
        score = np.asarray(ranker.predict(x)).ravel()
        top50 = set(map(int, bridge[np.argsort(-score, kind="stable")[:50]]))
        values.append(len(top50 & gold) / len(gold))
    return np.asarray(values, np.float32)

curve_rows = []
for target in (1200, 5000, 10000):
    use = min(target, len(expanded_l3_blocks))
    if use < 1000 or (curve_rows and use == curve_rows[-1]["groups"]):
        continue
    cat = _fit_expanded_l3(expanded_l3_blocks[:use], "QuerySoftMax", SEED+300+use)
    scores = _score_expanded_l3(cat)
    curve_rows.append({"groups":use, "objective":"QuerySoftMax",
                       "Recall@50":float(scores.mean()),
                       "minutes":round((time.perf_counter()-_l3_t0)/60,1)})
    print("L3 curve:", curve_rows[-1], flush=True)
    if target == 10000:
        expanded_l3_cat = cat
    else:
        del cat
    gc.collect(); torch.cuda.empty_cache()

# Сравнение topk делается при том же максимальном объёме обучения и тех же
# группах/признаках. Оба loss оптимизируют суррогат, решает Recall@50.
largest = curve_rows[-1]["groups"]
topk = _fit_expanded_l3(expanded_l3_blocks[:largest], "LambdaMART-topk", SEED+400)
topk_scores = _score_expanded_l3(topk)
curve_rows.append({"groups":largest, "objective":"LambdaMART-topk",
                   "Recall@50":float(topk_scores.mean()),
                   "minutes":round((time.perf_counter()-_l3_t0)/60,1)})
expanded_l3_xgb = topk
display(pd.DataFrame(curve_rows).round(4))
print("Проверочная группа уже просматривалась; кривая исследовательская.")
print("Для новых train-запросов linear обучался на тех же текстах; поэтому"
      " абсолютные цифры могут быть оптимистичны. answer.csv не создаётся.")


## 5. Дообучение BGE cross-encoder

Положительные тексты — выбранные пары train. Остальные кандидаты используются как слабые отрицательные примеры; LoRA обучается 3 эпохи. Оценки для проверки и benchmark считаются на двух GPU.


In [ ]:
# LoRA-дообучение BGE-reranker на выбранных парах train.
# Одна ячейка после опыта LambdaMART-topk (ячейка 2).
# Дообучаем BGE-reranker-v2-m3 на выбранных парах train и проверяем
# его добавку при неизменном пуле L2 top-750. Граф, история кликов и
# item_id не входят во вход cross-encoder. answer.csv НЕ создаётся.
import gc
import hashlib
import importlib.util
import json
import os
import random
import subprocess
import sys
import time
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from functools import lru_cache
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
import torch.nn.functional as F
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForSequenceClassification, AutoTokenizer

if importlib.util.find_spec("peft") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "peft>=0.13,<0.19"])
from peft import LoraConfig, PeftModel, TaskType, get_peft_model

# В некоторых Kaggle-образах одновременно стоят PEFT и устаревший
# torchao 0.10.0. PEFT проверяет его даже для обычных torch.nn.Linear и
# падает до начала LoRA. У нашей fp32-модели нет torchao-слоёв; отключаем
# только опциональный torchao-диспетчер, не меняя PyTorch или веса модели.
from peft.tuners.lora import torchao as _ce_peft_torchao
_ce_peft_torchao.is_torchao_available = lambda: False

needed = ["cases", "events", "excluded_texts", "items", "bge_items",
          "bge_query_text", "expanded_l3_xgb", "expanded_l3_hold",
          "BGE_SOURCE", "BGE_MAX_TOKENS", "WORK", "Q_COLS", "N_DEV",
          "N_HOLDOUT", "SEED", "norm", "DATA"]
assert all(name in globals() for name in needed), (
    "Сначала запустите основной ноутбук и ячейку LambdaMART-topk")
assert torch.cuda.is_available(), "Для этого опыта нужен Kaggle GPU"
assert len(expanded_l3_hold) == N_HOLDOUT == 500
assert all(len(row[0]) <= 750 for row in expanded_l3_hold)

# Меняйте эти значения перед началом нового эксперимента, не после просмотра
# результатов. 20 тысяч групп и четыре пары на группу дают ~80 тысяч
# прочитанных cross-encoder пар за эпоху. Если групп меньше, берём все.
ce_source_override = os.environ.get("AVITO_RERANKER_DIR")
ce_attached = [p.parent for p in Path("/kaggle/input").rglob("config.json")
               if "bge-reranker-v2-m3" in str(p.parent).lower()
               and ((p.parent / "model.safetensors").exists()
                    or (p.parent / "pytorch_model.bin").exists()
                    or (p.parent / "model.safetensors.index.json").exists())]
CE_SOURCE = (ce_source_override or
             (str(ce_attached[0]) if ce_attached else "BAAI/bge-reranker-v2-m3"))
print("Cross-encoder source:", CE_SOURCE, flush=True)
CE_GROUP_LIMIT = 20_000
CE_MAX_LENGTH = 192
CE_GROUP_BATCH = 2
CE_ACCUMULATE = 8
CE_EPOCHS = 3
CE_LR = 1e-4
CE_LR_SCHEDULE = (1e-4, 5e-5, 2e-5)
CE_INFER_BATCH = 24
CE_SEED = SEED + 1919
CE_RRF_WEIGHTS = (0.15, 0.30)  # только исследовательские сравнения
CE_START = time.perf_counter()
random.seed(CE_SEED)
np.random.seed(CE_SEED)
torch.manual_seed(CE_SEED)
torch.cuda.manual_seed_all(CE_SEED)


def ce_clean(value, limit):
    return "" if value is None or pd.isna(value) else str(value).strip()[:limit]


@lru_cache(maxsize=100_000)
def ce_item_text(pos):
    pos = int(pos)
    return ce_doc_text(items.item_title_raw.iat[pos],
                       items.item_infm_params_text.iat[pos],
                       items.item_description_raw.iat[pos])


def ce_doc_text(title, params, description):
    return (ce_clean(title, 220)
            + "\nПараметры: " + ce_clean(params, 260)
            + "\nОписание: " + ce_clean(description, 750))


# Используем положительные пары из всего train.parquet, включая объявления,
# которых уже нет в benchmark_items. Их текст присутствует в строке train.
# Полный набор выбранных корпусных item на тот же запрос всё равно исключён
# из негативов. Ни item_id, ни число выборов не подаются в cross-encoder.
blocked_texts = set(excluded_texts) | set(globals().get("prior_inspected", ()))
blocked_normalized = {norm(text, 160) for text in blocked_texts}
ce_gold_by_query = defaultdict(set)
for key, positives in events.items():
    raw_q = key[0]
    if raw_q in blocked_texts or norm(raw_q, 160) in blocked_normalized:
        continue
    ce_qtext = bge_query_text(dict(zip(Q_COLS, key))).strip()
    if ce_qtext:
        ce_gold_by_query[ce_qtext].update(map(int, positives))

ce_parquet = pq.ParquetFile(DATA / "train.parquet")
ce_all_queries = set()
for batch in ce_parquet.iter_batches(batch_size=25_000, columns=Q_COLS):
    rows = batch.to_pydict()
    for j in range(batch.num_rows):
        raw_q = rows["search_query"][j]
        if raw_q in blocked_texts or norm(raw_q, 160) in blocked_normalized:
            continue
        ce_qtext = bge_query_text({col: rows[col][j] for col in Q_COLS}).strip()
        if ce_qtext:
            ce_all_queries.add(ce_qtext)
ce_queries = sorted(ce_all_queries)
random.Random(CE_SEED).shuffle(ce_queries)
ce_queries = ce_queries[:CE_GROUP_LIMIT]
assert len(ce_queries) >= 1000, "Недостаточно обучающих запросов после исключения holdout"
ce_selected = set(ce_queries)
ce_docs_by_query = defaultdict(list)
ce_seen_pairs = set()
ce_train_cols = Q_COLS + ["item_id", "item_title_raw",
                          "item_infm_params_text", "item_description_raw",
                          "item_microcat_id"]
for batch in ce_parquet.iter_batches(batch_size=15_000, columns=ce_train_cols):
    rows = batch.to_pydict()
    for j in range(batch.num_rows):
        ce_qtext = bge_query_text({col: rows[col][j] for col in Q_COLS}).strip()
        if ce_qtext not in ce_selected or len(ce_docs_by_query[ce_qtext]) >= 8:
            continue
        pair = (ce_qtext, rows["item_id"][j])
        if pair in ce_seen_pairs:
            continue
        ce_seen_pairs.add(pair)
        document = ce_doc_text(rows["item_title_raw"][j],
                               rows["item_infm_params_text"][j],
                               rows["item_description_raw"][j])
        if document.strip():
            ce_docs_by_query[ce_qtext].append((document,
                                               rows["item_microcat_id"][j]))
ce_queries = [q for q in ce_queries if ce_docs_by_query[q]]
ce_gold = [ce_gold_by_query.get(q, set()) for q in ce_queries]
ce_pop = Counter(pos for gold in ce_gold for pos in gold)
print("Cross-encoder train: запросов", len(ce_queries),
      "| доступных уникальных выбранных текстов", sum(map(len, ce_docs_by_query.values())),
      "| corpus item с известным выбором", len(ce_pop), flush=True)

# Получаем средней сложности негативы из BGE top-150. Это лишь кандидаты,
# а не доказанно плохие объявления, поэтому их вклад в loss мал. Случайный
# item другой подкатегории даёт простой контраст; случайный item той же
# подкатегории даёт умеренный контраст без требования буквального совпадения.
ce_retrieval_path = WORK / (
    "ce_negatives_" + hashlib.sha256(
        (str(BGE_SOURCE) + str(CE_SEED) + "\n" + "\n".join(ce_queries))
        .encode("utf-8")).hexdigest()[:16] + ".npy")
if ce_retrieval_path.exists():
    ce_retrieved = np.load(ce_retrieval_path)
    assert ce_retrieved.shape == (len(ce_queries), 150)
    print("Негативы BGE загружены из кеша", flush=True)
else:
    ce_device_retrieval = "cuda:1" if torch.cuda.device_count() > 1 else "cuda:0"
    ce_encoder = SentenceTransformer(BGE_SOURCE, device=ce_device_retrieval)
    ce_encoder.max_seq_length = BGE_MAX_TOKENS
    ce_encoder.half().eval()
    ce_qvec = ce_encoder.encode(ce_queries, batch_size=64,
                                show_progress_bar=True,
                                normalize_embeddings=True)
    del ce_encoder
    gc.collect()
    torch.cuda.empty_cache()
    ce_device = torch.device(ce_device_retrieval)
    ce_iv = F.normalize(torch.as_tensor(np.asarray(bge_items),
                                         device=ce_device, dtype=torch.float16), dim=1)
    ce_retrieved = np.empty((len(ce_queries), 150), np.int32)
    with torch.inference_mode():
        for start in range(0, len(ce_queries), 128):
            q = torch.as_tensor(ce_qvec[start:start+128], device=ce_device,
                                dtype=torch.float16)
            q = F.normalize(q, dim=1)
            sims = q @ ce_iv.T
            ce_retrieved[start:start+len(q)] = torch.topk(
                sims, k=150, dim=1).indices.cpu().numpy()
            if (start // 128 + 1) % 20 == 0:
                print("BGE-поиск негативов:", min(start+128, len(ce_queries)),
                      "/", len(ce_queries), flush=True)
    np.save(ce_retrieval_path, ce_retrieved)
    del ce_iv, ce_qvec
    gc.collect()
    torch.cuda.empty_cache()


ce_rng = np.random.default_rng(CE_SEED)
ce_microcat = np.asarray(items.item_microcat_id)
ce_by_microcat = defaultdict(list)
for pos, ce_category in enumerate(ce_microcat):
    ce_by_microcat[ce_category].append(pos)
ce_groups = []
ce_missing_microcat = 0
for index, (query, gold) in enumerate(zip(ce_queries, ce_gold)):
    positives = ce_docs_by_query[query]
    positive_text, positive_cat = positives[int(ce_rng.integers(len(positives)))]
    # При попадании нескольких выбранных item в top-150 пропускаем их все.
    hard_choices = [int(pos) for pos in ce_retrieved[index, 40:150]
                    if int(pos) not in gold]
    if not hard_choices:
        continue
    hard = int(ce_rng.choice(hard_choices))
    if positive_cat is None or pd.isna(positive_cat):
        positive_cat = ce_microcat[hard]
    same_choices = ce_by_microcat.get(positive_cat)
    if not same_choices:
        # Положительный item может отсутствовать в benchmark_items вместе со
        # всей своей подкатегорией. Тогда берём подкатегорию BGE-соседа, а не
        # вызываем choice() на пустом списке.
        ce_missing_microcat += 1
        same_choices = ce_by_microcat.get(ce_microcat[hard])
    if not same_choices:
        same_choices = hard_choices
    same = -1
    for _ in range(20):
        candidate = int(ce_rng.choice(same_choices))
        if candidate not in gold and candidate != hard:
            same = candidate
            break
    if same < 0:
        same = next((candidate for candidate in hard_choices
                     if candidate != hard), hard)
    easy = -1
    for _ in range(30):
        candidate = int(ce_rng.integers(len(items)))
        if candidate not in gold and ce_microcat[candidate] != positive_cat:
            easy = candidate
            break
    if easy < 0:
        continue
    ce_groups.append((query, positive_text, easy, same, hard))
assert len(ce_groups) >= 1000
ce_probe = random.Random(CE_SEED + 1).sample(ce_groups, min(3, len(ce_groups)))
print("Групп для LoRA:", len(ce_groups),
      "| подкатегорий train вне корпуса:", ce_missing_microcat,
      "| примеры запросов:", [g[0][:50] for g in ce_probe], flush=True)

# Для XLM-R типы query/value соответствуют self-attention. Классификатор
# обучается вместе с LoRA. Основные веса остаются fp32, операции идут в
# autocast fp16: это избегает прежней ошибки Half vs Float.
ce_key = hashlib.sha256(json.dumps({
    "source": CE_SOURCE, "seed": CE_SEED, "groups": len(ce_groups),
    "q_hash": hashlib.sha256("\n".join(ce_queries).encode()).hexdigest(),
    "gold_hash": hashlib.sha256("\n".join(
        ",".join(map(str, sorted(gold))) for gold in ce_gold).encode()).hexdigest(),
    "positive_text_hash": hashlib.sha256("\n".join(
        ce_docs_by_query[q][0][0] for q in ce_queries).encode()).hexdigest(),
    "corpus_ends": [str(items.item_id.iat[0]), str(items.item_id.iat[-1])],
    "max_len": CE_MAX_LENGTH, "lr_schedule": CE_LR_SCHEDULE,
    "epochs": CE_EPOCHS,
}, sort_keys=True).encode()).hexdigest()[:16]
ce_adapter_dir = WORK / f"ce_lora_{ce_key}"
ce_adapter_dir.mkdir(parents=True, exist_ok=True)
ce_tokenizer = AutoTokenizer.from_pretrained(CE_SOURCE, use_fast=True)
if ((ce_adapter_dir / "adapter_config.json").exists()
        and any((ce_adapter_dir / name).exists()
                for name in ("adapter_model.safetensors", "adapter_model.bin"))):
    print("Загружаем ранее дообученный LoRA:", ce_adapter_dir, flush=True)
else:
    ce_base = AutoModelForSequenceClassification.from_pretrained(
        CE_SOURCE, num_labels=1, torch_dtype=torch.float32)
    ce_base.gradient_checkpointing_enable()
    ce_base.enable_input_require_grads()
    # Сохраняем адаптер после каждой эпохи. При обрыве Kaggle следующая
    # попытка продолжит с последней завершённой эпохи без повторного BGE.
    completed = [epoch for epoch in range(1, CE_EPOCHS + 1)
                 if ((ce_adapter_dir / f"epoch_{epoch}" / "adapter_config.json").exists()
                     and any((ce_adapter_dir / f"epoch_{epoch}" / name).exists()
                             for name in ("adapter_model.safetensors",
                                          "adapter_model.bin")))]
    start_epoch = max(completed, default=0)
    if start_epoch:
        ce_model = PeftModel.from_pretrained(
            ce_base, ce_adapter_dir / f"epoch_{start_epoch}",
            is_trainable=True).to("cuda:0")
        print("Продолжаем LoRA после эпохи", start_epoch, flush=True)
    else:
        ce_config = LoraConfig(task_type=TaskType.SEQ_CLS,
                               r=8, lora_alpha=16, lora_dropout=0.05,
                               target_modules=["query", "value"],
                               modules_to_save=["classifier"])
        ce_model = get_peft_model(ce_base, ce_config).to("cuda:0")
    ce_model.print_trainable_parameters()
    ce_optimizer = torch.optim.AdamW(
        (p for p in ce_model.parameters() if p.requires_grad),
        lr=CE_LR, weight_decay=0.01)
    ce_scaler = torch.amp.GradScaler("cuda")
    ce_model.train()
    ce_optimizer.zero_grad(set_to_none=True)
    for epoch in range(start_epoch, CE_EPOCHS):
        for group in ce_optimizer.param_groups:
            group["lr"] = CE_LR_SCHEDULE[epoch]
        ce_order = np.random.default_rng(
            CE_SEED + 2 + epoch).permutation(len(ce_groups))
        ce_steps = 0
        ce_running = 0.0
        print(f"Начало LoRA epoch {epoch+1}/{CE_EPOCHS} | "
              f"lr={CE_LR_SCHEDULE[epoch]:.1e}", flush=True)
        for offset in range(0, len(ce_order), CE_GROUP_BATCH):
            batch_ids = ce_order[offset:offset+CE_GROUP_BATCH]
            queries_batch, items_batch = [], []
            for group_idx in batch_ids:
                q, pos_text, easy, same, hard = ce_groups[int(group_idx)]
                queries_batch.extend([q] * 4)
                items_batch.extend([pos_text, ce_item_text(easy),
                                    ce_item_text(same), ce_item_text(hard)])
            encoded = ce_tokenizer(queries_batch, items_batch,
                                    padding=True, truncation="only_second",
                                    max_length=CE_MAX_LENGTH,
                                    return_tensors="pt")
            encoded = {k: v.to("cuda:0", non_blocking=True)
                       for k, v in encoded.items()}
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                logits = ce_model(**encoded).logits.float().view(-1, 4)
                # Пары с семантически близким объявлением могут быть ложными
                # негативами: их вес в три раза меньше простого негатива.
                diffs = logits[:, :1] - logits[:, 1:]
                loss = (F.softplus(-diffs[:, 0]).mean()
                        + 0.35*F.softplus(-diffs[:, 1]).mean()
                        + 0.25*F.softplus(-diffs[:, 2]).mean()) / 1.60
            ce_scaler.scale(loss / CE_ACCUMULATE).backward()
            ce_running += float(loss.detach())
            ce_steps += 1
            if ce_steps % CE_ACCUMULATE == 0 or offset+CE_GROUP_BATCH >= len(ce_order):
                ce_scaler.unscale_(ce_optimizer)
                torch.nn.utils.clip_grad_norm_(ce_model.parameters(), 1.0)
                ce_scaler.step(ce_optimizer)
                ce_scaler.update()
                ce_optimizer.zero_grad(set_to_none=True)
            if ce_steps % 500 == 0:
                print(f"LoRA epoch {epoch+1}/{CE_EPOCHS}: "
                      f"groups {min(offset+CE_GROUP_BATCH, len(ce_order))}/{len(ce_order)} "
                      f"loss {ce_running/500:.4f} "
                      f"elapsed {(time.perf_counter()-CE_START)/60:.1f} min",
                      flush=True)
                ce_running = 0.0
        epoch_dir = ce_adapter_dir / f"epoch_{epoch+1}"
        ce_model.save_pretrained(epoch_dir)
        print("Эпоха сохранена:", epoch_dir, flush=True)
    ce_model.save_pretrained(ce_adapter_dir)
    ce_tokenizer.save_pretrained(ce_adapter_dir)
    del ce_model, ce_base, ce_optimizer, ce_scaler
    gc.collect()
    torch.cuda.empty_cache()
    print("LoRA сохранена:", ce_adapter_dir, flush=True)


# Независимые копии модели для GPU 0/1. Запись частичных результатов отдельно
# по GPU позволяет продолжить после обрыва Kaggle, сохраняя порядок запросов.
def ce_score_worker(gpu_index, case_indices):
    device = f"cuda:{gpu_index}"
    cache_file = WORK / f"ce_scores_{ce_key}_gpu{gpu_index}.npy"
    if cache_file.exists():
        scores = np.load(cache_file)
        assert scores.shape == (N_HOLDOUT, 750)
    else:
        scores = np.full((N_HOLDOUT, 750), np.nan, np.float32)
    tokenizer = AutoTokenizer.from_pretrained(ce_adapter_dir, use_fast=True)
    base = AutoModelForSequenceClassification.from_pretrained(
        CE_SOURCE, num_labels=1, torch_dtype=torch.float32)
    model = PeftModel.from_pretrained(base, ce_adapter_dir).to(device).eval()
    fresh = 0
    for j in case_indices:
        bridge, _, _ = expanded_l3_hold[j]
        n = len(bridge)
        if np.isfinite(scores[j, :n]).all():
            continue
        query = bge_query_text(cases[N_DEV+j]["query"])
        values = np.empty(n, np.float32)
        with torch.inference_mode():
            for start in range(0, n, CE_INFER_BATCH):
                sub = bridge[start:start+CE_INFER_BATCH]
                batch = tokenizer([query]*len(sub),
                                  [ce_item_text(pos) for pos in sub],
                                  padding=True, truncation="only_second",
                                  max_length=CE_MAX_LENGTH,
                                  return_tensors="pt")
                batch = {k: v.to(device, non_blocking=True)
                         for k, v in batch.items()}
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    logits = model(**batch).logits.float().view(-1)
                values[start:start+len(sub)] = logits.cpu().numpy()
        scores[j, :n] = values
        fresh += 1
        if fresh % 10 == 0:
            temp_file = cache_file.with_name(cache_file.stem + ".tmp.npy")
            np.save(temp_file, scores)
            os.replace(temp_file, cache_file)
            print(f"Cross GPU{gpu_index}: {fresh}/{len(case_indices)} новых; "
                  f"общее время {(time.perf_counter()-CE_START)/60:.1f} мин",
                  flush=True)
    temp_file = cache_file.with_name(cache_file.stem + ".tmp.npy")
    np.save(temp_file, scores)
    os.replace(temp_file, cache_file)
    del model, base
    gc.collect()
    torch.cuda.empty_cache()
    return scores


ce_gpus = min(torch.cuda.device_count(), 2)
ce_shards = [list(range(gpu, N_HOLDOUT, ce_gpus)) for gpu in range(ce_gpus)]
ce_infer_start = time.perf_counter()
with ThreadPoolExecutor(max_workers=ce_gpus) as executor:
    ce_parts = list(executor.map(ce_score_worker, range(ce_gpus), ce_shards))
ce_infer_minutes = (time.perf_counter()-ce_infer_start)/60
ce_scores = np.full((N_HOLDOUT, 750), np.nan, np.float32)
for part in ce_parts:
    mask = np.isfinite(part)
    ce_scores[mask] = part[mask]
del ce_parts
for j, (bridge, _, _) in enumerate(expanded_l3_hold):
    assert np.isfinite(ce_scores[j, :len(bridge)]).all(), f"Неполные оценки query {j}"


def ce_ranks(values):
    order = np.argsort(-np.asarray(values), kind="stable")
    ranks = np.empty(len(order), np.int32)
    ranks[order] = np.arange(1, len(order)+1)
    return ranks


ce_recall = defaultdict(list)
ce_picks = defaultdict(list)
for j, (bridge, base_x, gold) in enumerate(expanded_l3_hold):
    n = len(bridge)
    xgb_score = np.asarray(expanded_l3_xgb.predict(base_x), np.float32).ravel()
    cross_score = ce_scores[j, :n]
    rank_xgb, rank_cross = ce_ranks(xgb_score), ce_ranks(cross_score)
    methods = {"LambdaMART": rank_xgb, "fine-tuned cross": rank_cross}
    for weight in CE_RRF_WEIGHTS:
        fused = ((1-weight)/(60.0+rank_xgb)
                 + weight/(60.0+rank_cross))
        methods[f"RRF cross={weight:.2f}"] = ce_ranks(fused)
    for name, ranks in methods.items():
        chosen = set(map(int, bridge[np.argsort(ranks)[:50]]))
        ce_picks[name].append(chosen)
        ce_recall[name].append(len(chosen & gold)/len(gold))

base_name = "LambdaMART"
base_hit = [ce_picks[base_name][j] & expanded_l3_hold[j][2]
            for j in range(N_HOLDOUT)]
summary = []
for name in ce_recall:
    found = sum(len((ce_picks[name][j] & expanded_l3_hold[j][2])-base_hit[j])
                for j in range(N_HOLDOUT))
    lost = sum(len(base_hit[j]-(ce_picks[name][j] & expanded_l3_hold[j][2]))
               for j in range(N_HOLDOUT))
    summary.append({"method": name,
                    "Recall@50": float(np.mean(ce_recall[name])),
                    "new_gold": found, "lost_gold": lost})
display(pd.DataFrame(summary).round(4))

# Item без выбора в доступном train — отдельный срез. Предсказание модели
# использует только текст, но изменение hit rate на cold item важно видеть.
ce_warm_all = {int(pos) for selected in events.values() for pos in selected}
slice_rows = []
for slice_name, is_member in (("cold item", lambda pos: pos not in ce_warm_all),
                              ("warm item", lambda pos: pos in ce_warm_all)):
    count = sum(is_member(pos) for _, _, gold in expanded_l3_hold for pos in gold)
    row = {"slice": slice_name, "gold": count}
    for name, picks in ce_picks.items():
        hits = sum(is_member(pos) and pos in picks[j]
                   for j, (_, _, gold) in enumerate(expanded_l3_hold)
                   for pos in gold)
        row[name] = hits/count if count else np.nan
    slice_rows.append(row)
display(pd.DataFrame(slice_rows).round(4))

# Интервал описывает разброс по этим 500 запросам; это НЕ независимое
# подтверждение, поскольку архитектура каскада уже выбиралась с их участием.
ce_rng_ci = np.random.default_rng(CE_SEED + 33)
for name in ce_recall:
    if name == base_name:
        continue
    delta = np.asarray(ce_recall[name])-np.asarray(ce_recall[base_name])
    draws = ce_rng_ci.integers(0, len(delta), size=(2000, len(delta)))
    ci = np.quantile(delta[draws].mean(axis=1), [0.025, 0.975])
    print(name, "Δ vs LambdaMART =", round(float(delta.mean()), 4),
          "| descriptive 95%", np.round(ci, 4), flush=True)
print("Инференс на", ce_gpus, "GPU, мин:", round(ce_infer_minutes, 1),
      "| весь эксперимент, мин:", round((time.perf_counter()-CE_START)/60, 1))
print("Holdout ранее просматривался; RRF веса нельзя выбирать по этой таблице.")
print("answer.csv не создаётся.")


## 6. Предсказание и проверка CSV

Для benchmark повторяем тот же каскад, соединяем ранги LambdaMART и BGE с фиксированными весами 0,85/0,15, записываем CSV и проверяем идентификаторы объявлений.


In [ ]:
# Полное предсказание benchmark, проверка и запись answer.csv.
# Одна ячейка в конце текущего Kaggle-ноутбука ПОСЛЕ обучения LoRA.
# Каскад: pool_both -> CatBoost L1 top-1500 -> CatBoost L2 top-750
# -> LambdaMART + дообученный BGE cross-encoder -> answer.csv.
# Граф и глобальная история выборов в финальном балле не используются.
# Веса смеси зафиксированы до просмотра benchmark-разметки.
import gc
import hashlib
import json
import os
import re
import time
from concurrent.futures import ThreadPoolExecutor
from itertools import chain
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
from peft import PeftModel
from peft.tuners.lora import torchao as _answer_peft_torchao
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import normalize as l2_normalize
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# Не используем torchao-слои. В образе Kaggle может быть torchao 0.10.0,
# который PEFT ошибочно проверяет даже для обычных torch.nn.Linear.
_answer_peft_torchao.is_torchao_available = lambda: False

required = ["cases", "items", "DATA", "WORK", "Q_COLS", "N_DEV", "N_HOLDOUT",
            "N_ITEMS", "STAGE2_POOL", "STAGE2_ALPHA", "FINAL_K", "BRIDGE_K",
            "POOL_LIMIT", "LEXICAL_RESERVE", "PREFIX_PER_EXPRESSION",
            "VECTOR_GLOBAL_K", "VECTOR_NEAR_K", "BGE_SOURCE", "BGE_MAX_TOKENS",
            "BGE_BATCH", "bge_items", "bge_queries", "bge_query_text",
            "tfidf", "lsa", "lsa_items", "linear_items", "X_val_q",
            "lsa_queries", "linear_queries", "make_broad_pool", "query_text",
            "encode_csr", "nearby_indices", "cheap_scores", "ranks_desc", "norm",
            "feature_matrix", "final_features", "stage2_ranker", "l2_ranker",
            "expanded_l3_xgb", "CE_SOURCE", "ce_adapter_dir", "CE_MAX_LENGTH",
            "CE_INFER_BATCH", "ce_item_text", "DEVICE"]
assert all(name in globals() for name in required), (
    "Сначала выполните основной каскад, LambdaMART-topk и обучение BGE LoRA")

# Ранние версии обучающей ячейки перезаписывали функцию query_text строкой.
# Восстанавливаем исходную реализацию до векторизации benchmark-запросов.
if not callable(query_text):
    def query_text(query):
        return (norm(query.get("search_query")) + " "
                + norm(query.get("search_infm_params_text")))
    print("Восстановлена функция query_text", flush=True)

# В ранней обучающей ячейке цикл по подкатегориям мог затереть глобальный
# массив item-категорий одним числом. feature_matrix индексирует этот массив.
if ("category" not in globals() or np.ndim(category) != 1
        or len(category) != len(items)):
    category = pd.to_numeric(items.item_category_id, errors="coerce")\
        .fillna(-1).to_numpy(np.int32)
    print("Восстановлен массив категорий объявлений", flush=True)

assert STAGE2_POOL == "pool_both" and FINAL_K == 1500 and BRIDGE_K == 750
assert torch.cuda.is_available(), "Для полного cross-encoder нужен Kaggle GPU"
WORK = Path(WORK)
ce_adapter_dir = Path(ce_adapter_dir)
adapter_file = next((ce_adapter_dir / name for name in
                     ("adapter_model.safetensors", "adapter_model.bin")
                     if (ce_adapter_dir / name).exists()), None)
assert (ce_adapter_dir / "adapter_config.json").exists() and adapter_file is not None, (
    "LoRA ещё не сохранена. Дождитесь завершения обучающей ячейки")

ANSWER_CROSS_WEIGHT = 0.15  # фиксированная консервативная RRF-добавка
ANSWER_TOP = 50
_answer_t0 = time.perf_counter()

# Идентификаторы читаются из benchmark без преобразования регистра или порядка.
benchmark_file = Path(DATA) / "benchmark_queries.parquet"
benchmark = pd.read_parquet(benchmark_file)
assert set(["query_id", *Q_COLS]).issubset(benchmark.columns)
benchmark_ids = benchmark.query_id.astype(str).tolist()
benchmark_queries = [dict(zip(Q_COLS, row)) for row in
                     benchmark[Q_COLS].itertuples(index=False, name=None)]
assert len(benchmark_ids) == 2452 and len(set(benchmark_ids)) == len(benchmark_ids)
assert all(len(query_id) == 16 for query_id in benchmark_ids)
benchmark_sha = hashlib.sha256(benchmark_file.read_bytes()).hexdigest()
adapter_sha = hashlib.sha256(adapter_file.read_bytes()).hexdigest()
benchmark_begin = N_DEV + N_HOLDOUT
N_BENCH = len(benchmark_ids)


def answer_query_matches(left, right):
    return all(str(left.get(key)) == str(right.get(key)) for key in Q_COLS)


# Ячейка допускает повторный запуск. Если предыдущая попытка остановилась
# посреди подготовки benchmark, обрезаем лишь дописанные запросы и векторы.
if len(cases) not in (benchmark_begin, benchmark_begin + N_BENCH):
    raise RuntimeError(f"Неожиданная длина cases: {len(cases)}")
if len(cases) == benchmark_begin + N_BENCH:
    assert all(answer_query_matches(cases[benchmark_begin+j]["query"], q)
               for j, q in enumerate(benchmark_queries)), (
        "В cases уже другой benchmark или иной порядок запросов")
    bench_ready = (X_val_q.shape[0] >= benchmark_begin + N_BENCH
                   and len(lsa_queries) >= benchmark_begin + N_BENCH
                   and len(linear_queries) >= benchmark_begin + N_BENCH
                   and len(bge_queries) >= benchmark_begin + N_BENCH
                   and all(all(name in case for name in
                               ("pool_both", "pool_all", "lex", "lsa", "linear",
                                "prefixes", "cheap_all", "bge_all",
                                STAGE2_POOL + "_rank"))
                           for case in cases[benchmark_begin:]))
else:
    bench_ready = False

if not bench_ready:
    cases = cases[:benchmark_begin]
    X_val_q = X_val_q[:benchmark_begin]
    lsa_queries = lsa_queries[:benchmark_begin]
    linear_queries = linear_queries[:benchmark_begin]
    bge_queries = bge_queries[:benchmark_begin]
    if "val_q_docs" in globals():
        val_q_docs = val_q_docs[:benchmark_begin]
    cases.extend({"query": q, "gold": set()} for q in benchmark_queries)

    # Широкий пул: тот же индекс и те же ограничения, что на проверке.
    broad_signature = {
        "query_sha": benchmark_sha,
        "item_first_last": [str(items.item_id.iat[0]),
                            str(items.item_id.iat[-1])],
        "n_items": N_ITEMS,
        "pool_limit": POOL_LIMIT,
        "lexical_reserve": LEXICAL_RESERVE,
        "prefix_per_expression": PREFIX_PER_EXPRESSION,
    }
    broad_cache = WORK / f"benchmark_broad_{benchmark_sha[:16]}.joblib"
    cached = joblib.load(broad_cache) if broad_cache.exists() else None
    if (isinstance(cached, dict) and cached.get("signature") == broad_signature
            and len(cached.get("rows", ())) == N_BENCH):
        broad_rows = cached["rows"]
        print("Широкий пул загружен из кеша", flush=True)
    else:
        broad_rows = []
        for j, q in enumerate(benchmark_queries, 1):
            broad_rows.append(make_broad_pool(q))
            if j % 200 == 0 or j == N_BENCH:
                print("Широкий пул:", j, "/", N_BENCH, flush=True)
        joblib.dump({"signature": broad_signature, "rows": broad_rows},
                    broad_cache, compress=3)
    for case, (base, lex, prefixes) in zip(cases[benchmark_begin:], broad_rows):
        case.update(base=base, lex=lex, prefixes=prefixes)

    # Иногда в исследовательской сессии переменная tfidf была случайно
    # заменена массивом. Восстанавливаем только векторизатор.
    if not hasattr(tfidf, "transform"):
        assert "tfidf_path" in globals()
        tfidf = joblib.load(tfidf_path)
    new_docs = [query_text(q) for q in benchmark_queries]
    new_X = tfidf.transform(new_docs).tocsr()
    X_val_q = sp.vstack((X_val_q, new_X), format="csr")
    if "val_q_docs" in globals():
        val_q_docs.extend(new_docs)
    new_lsa = l2_normalize(lsa.transform(new_X).astype(np.float32))
    new_linear = encode_csr(new_X, "query")
    lsa_queries = np.vstack((lsa_queries, new_lsa))
    linear_queries = np.vstack((linear_queries, new_linear))

    def answer_vector_sources(item_vectors, query_vectors, label):
        iv = torch.as_tensor(np.asarray(item_vectors), dtype=torch.float32,
                             device=DEVICE)
        result = []
        with torch.no_grad():
            for j, (q, q_np) in enumerate(zip(benchmark_queries, query_vectors), 1):
                if not np.any(q_np):
                    result.append(np.empty(0, np.int32))
                    continue
                scores = iv @ torch.as_tensor(q_np, device=DEVICE,
                                              dtype=torch.float32)
                selected = []
                if VECTOR_GLOBAL_K:
                    selected.extend(torch.topk(
                        scores, k=min(VECTOR_GLOBAL_K, N_ITEMS)).indices.cpu().tolist())
                near = nearby_indices(q.get("search_location_id"))
                if len(near):
                    near_gpu = torch.as_tensor(near.astype(np.int64), device=DEVICE)
                    local = torch.topk(scores[near_gpu],
                                       k=min(VECTOR_NEAR_K, len(near))).indices
                    selected.extend(near_gpu[local].cpu().tolist())
                unique = np.asarray(list(dict.fromkeys(selected)), np.int32)
                if len(unique):
                    order = torch.argsort(scores[
                        torch.as_tensor(unique, device=DEVICE)],
                        descending=True).cpu().numpy()
                    unique = unique[order]
                result.append(unique)
                if j % 250 == 0 or j == N_BENCH:
                    print(label, j, "/", N_BENCH, flush=True)
        del iv
        torch.cuda.empty_cache()
        return result

    lsa_source = answer_vector_sources(lsa_items, new_lsa, "LSA")
    linear_source = answer_vector_sources(linear_items, new_linear, "Linear")
    for case, ls, lin in zip(cases[benchmark_begin:], lsa_source, linear_source):
        case["lsa"], case["linear"] = ls, lin
        case["pool_both"] = np.unique(np.concatenate((case["base"], ls, lin)))
        case["pool_all"] = case["pool_both"]
    del lsa_source, linear_source
    gc.collect()

    # Корпусные BGE-векторы уже были рассчитаны; кодируем лишь 2452 запроса.
    bge_cache = WORK / f"benchmark_bge_{benchmark_sha[:16]}_len{BGE_MAX_TOKENS}.npy"
    bge_meta = bge_cache.with_suffix(".json")
    bge_signature = {"query_sha": benchmark_sha, "source": str(BGE_SOURCE),
                     "max_tokens": BGE_MAX_TOKENS}
    if (bge_cache.exists() and bge_meta.exists()
            and json.loads(bge_meta.read_text()) == bge_signature):
        new_bge = np.load(bge_cache)
        assert new_bge.shape == (N_BENCH, bge_items.shape[1])
        print("BGE-векторы запросов загружены из кеша", flush=True)
    else:
        bge_device = "cuda:1" if torch.cuda.device_count() > 1 else "cuda:0"
        bge_encoder = SentenceTransformer(BGE_SOURCE, device=bge_device)
        bge_encoder.max_seq_length = BGE_MAX_TOKENS
        bge_encoder.half().eval()
        new_bge = bge_encoder.encode(
            [bge_query_text(q) for q in benchmark_queries], batch_size=BGE_BATCH,
            normalize_embeddings=True, convert_to_numpy=True,
            show_progress_bar=True).astype(np.float32)
        np.save(bge_cache, new_bge)
        bge_meta.write_text(json.dumps(bge_signature), encoding="utf-8")
        del bge_encoder
        gc.collect()
        torch.cuda.empty_cache()
    bge_queries = np.vstack((bge_queries, new_bge))

    # Эти величины читает feature_matrix L1 и final_features L2.
    item_gpu = torch.as_tensor(np.asarray(bge_items), device=DEVICE,
                               dtype=torch.float16)
    with torch.no_grad():
        for j, case in enumerate(cases[benchmark_begin:], 1):
            ids = case[STAGE2_POOL]
            case["cheap_all"] = cheap_scores(case, ids)
            gathered = item_gpu[torch.as_tensor(ids.astype(np.int64),
                                                 device=DEVICE)]
            q_gpu = torch.as_tensor(new_bge[j-1], device=DEVICE,
                                    dtype=torch.float32)
            case["bge_all"] = (gathered.float() @ q_gpu).cpu().numpy()
            case[STAGE2_POOL + "_rank"] = (
                ranks_desc(case["cheap_all"]), ranks_desc(case["bge_all"]))
            if j % 250 == 0 or j == N_BENCH:
                print("Балл широкого пула:", j, "/", N_BENCH, flush=True)
    del item_gpu
    torch.cuda.empty_cache()
    assert (X_val_q.shape[0] == len(lsa_queries) == len(linear_queries)
            == len(bge_queries) == len(cases))

print("Benchmark подготовлен:", N_BENCH,
      "| средний широкий пул:", round(float(np.mean([
          len(case[STAGE2_POOL]) for case in cases[benchmark_begin:]])), 1),
      flush=True)

# Если прежняя ячейка освободила эти небольшие тензоры, восстановим их.
if "lsa_gpu" not in globals():
    lsa_gpu = torch.as_tensor(np.asarray(lsa_items), device=DEVICE,
                              dtype=torch.float16)
if "linear_gpu" not in globals():
    linear_gpu = torch.as_tensor(np.asarray(linear_items), device=DEVICE,
                                 dtype=torch.float16)

# L1/L2 и LambdaMART для всех запросов. Кеш привязан к объектам обученных
# моделей внутри этой сессии, так что после перезапуска kernel он не смешает
# веса новой модели со старыми предсказаниями.
upstream_signature = {
    "query_sha": benchmark_sha,
    "stage2_id": id(stage2_ranker), "l2_id": id(l2_ranker),
    "xgb_id": id(expanded_l3_xgb),
    "n_items": len(items), "bridge_k": BRIDGE_K,
}
upstream_key = hashlib.sha256(json.dumps(
    upstream_signature, sort_keys=True).encode()).hexdigest()[:16]
upstream_cache = WORK / f"benchmark_to750_xgb_{upstream_key}.npz"
if upstream_cache.exists():
    saved = np.load(upstream_cache)
    bridge_pos = saved["bridge_pos"]
    bridge_len = saved["bridge_len"]
    xgb_scores = saved["xgb_scores"]
    l1_top50 = saved["l1_top50"]
    assert bridge_pos.shape == xgb_scores.shape == (N_BENCH, BRIDGE_K)
    assert bridge_len.shape == (N_BENCH,) and l1_top50.shape == (N_BENCH, 50)
    print("Top-750 и LambdaMART загружены из кеша", flush=True)
else:
    bridge_pos = np.full((N_BENCH, BRIDGE_K), -1, np.int32)
    bridge_len = np.zeros(N_BENCH, np.int32)
    xgb_scores = np.full((N_BENCH, BRIDGE_K), np.nan, np.float32)
    l1_top50 = np.full((N_BENCH, 50), -1, np.int32)
    for j, i in enumerate(range(benchmark_begin, len(cases))):
        pool = cases[i][STAGE2_POOL]
        if not len(pool):
            continue
        l1_score = np.asarray(stage2_ranker.predict(feature_matrix(i, pool)),
                              np.float32).ravel()
        l1 = pool[np.argsort(-l1_score, kind="stable")[:FINAL_K]]
        l1_top50[j, :min(50, len(l1))] = l1[:50]
        x2 = final_features(i, l1, np.arange(1, len(l1)+1))
        l2_score = np.asarray(l2_ranker.predict(x2), np.float32).ravel()
        at = np.argsort(-l2_score, kind="stable")[:BRIDGE_K]
        bridge = l1[at]
        n = len(bridge)
        bridge_len[j] = n
        bridge_pos[j, :n] = bridge
        x3 = np.column_stack((x2[at], np.log1p(np.arange(1, n+1)))).astype(np.float32)
        xgb_scores[j, :n] = np.asarray(expanded_l3_xgb.predict(x3),
                                       np.float32).ravel()
        if (j+1) % 100 == 0 or j+1 == N_BENCH:
            print("L1 -> 1500 -> L2 -> 750 -> LambdaMART:",
                  j+1, "/", N_BENCH, flush=True)
    np.savez_compressed(upstream_cache, bridge_pos=bridge_pos,
                        bridge_len=bridge_len, xgb_scores=xgb_scores,
                        l1_top50=l1_top50)

# Cross-encoder читает все top-750. Каждый GPU пишет свой кеш, так что
# повторный запуск досчитает только оставшиеся пары.
cross_key = hashlib.sha256(json.dumps({
    "benchmark_sha": benchmark_sha, "adapter_sha": adapter_sha,
    "upstream_key": upstream_key, "max_len": CE_MAX_LENGTH,
}, sort_keys=True).encode()).hexdigest()[:16]


def answer_cross_worker(gpu_id, query_indices):
    device = f"cuda:{gpu_id}"
    cache_file = WORK / f"benchmark_cross_{cross_key}_gpu{gpu_id}.npy"
    scores = (np.load(cache_file) if cache_file.exists() else
              np.full((N_BENCH, BRIDGE_K), np.nan, np.float32))
    assert scores.shape == (N_BENCH, BRIDGE_K)
    fresh = [j for j in query_indices
             if bridge_len[j] and not np.isfinite(scores[j, :bridge_len[j]]).all()]
    if not fresh:
        return scores
    tokenizer = AutoTokenizer.from_pretrained(ce_adapter_dir, use_fast=True)
    base = AutoModelForSequenceClassification.from_pretrained(
        CE_SOURCE, num_labels=1, torch_dtype=torch.float32)
    cross_model = PeftModel.from_pretrained(base, ce_adapter_dir).to(device).eval()
    for count, j in enumerate(fresh, 1):
        n = int(bridge_len[j])
        qtext = bge_query_text(benchmark_queries[j])
        ids = bridge_pos[j, :n]
        values = np.empty(n, np.float32)
        with torch.inference_mode():
            for start in range(0, n, CE_INFER_BATCH):
                sub = ids[start:start+CE_INFER_BATCH]
                encoded = tokenizer([qtext]*len(sub),
                                    [ce_item_text(pos) for pos in sub],
                                    padding=True, truncation="only_second",
                                    max_length=CE_MAX_LENGTH,
                                    return_tensors="pt")
                encoded = {name: tensor.to(device, non_blocking=True)
                           for name, tensor in encoded.items()}
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    logits = cross_model(**encoded).logits.float().view(-1)
                values[start:start+len(sub)] = logits.cpu().numpy()
        scores[j, :n] = values
        if count % 20 == 0 or count == len(fresh):
            temporary = cache_file.with_name(cache_file.stem + ".tmp.npy")
            np.save(temporary, scores)
            os.replace(temporary, cache_file)
            print(f"Cross GPU{gpu_id}: {count}/{len(fresh)} новых | "
                  f"прошло {(time.perf_counter()-_answer_t0)/60:.1f} мин",
                  flush=True)
    del cross_model, base
    gc.collect()
    torch.cuda.empty_cache()
    return scores


gpu_count = min(torch.cuda.device_count(), 2)
shards = [list(range(gpu, N_BENCH, gpu_count)) for gpu in range(gpu_count)]
cross_start = time.perf_counter()
with ThreadPoolExecutor(max_workers=gpu_count) as executor:
    cross_parts = list(executor.map(answer_cross_worker, range(gpu_count), shards))
cross_scores = np.full((N_BENCH, BRIDGE_K), np.nan, np.float32)
for part in cross_parts:
    finite = np.isfinite(part)
    cross_scores[finite] = part[finite]
del cross_parts
cross_minutes = (time.perf_counter()-cross_start)/60
for j, n in enumerate(bridge_len):
    assert np.isfinite(cross_scores[j, :n]).all(), f"Cross-encoder не оценил запрос {j}"


def answer_ranks(score):
    order = np.argsort(-np.asarray(score), kind="stable")
    ranks = np.empty(len(order), np.int32)
    ranks[order] = np.arange(1, len(order)+1)
    return ranks


all_predictions = []
changed_vs_xgb = 0
fallback_count = 0
for j in range(N_BENCH):
    n = int(bridge_len[j])
    bridge = bridge_pos[j, :n]
    if n:
        r_xgb = answer_ranks(xgb_scores[j, :n])
        r_cross = answer_ranks(cross_scores[j, :n])
        mix = ((1-ANSWER_CROSS_WEIGHT)/(60.0+r_xgb)
               + ANSWER_CROSS_WEIGHT/(60.0+r_cross))
        chosen = bridge[np.argsort(-mix, kind="stable")[:ANSWER_TOP]].tolist()
        xgb_top = set(map(int, bridge[np.argsort(r_xgb)[:ANSWER_TOP]]))
        changed_vs_xgb += set(map(int, chosen)) != xgb_top
    else:
        chosen = []
    if len(chosen) < ANSWER_TOP:
        fallback_count += 1
        seen = set(map(int, chosen))
        # Сначала остальные результаты L1, затем общий корпус лишь если весь
        # широкий пул был пуст. Никакой привязки к query_id здесь нет.
        for pos in chain(l1_top50[j], range(N_ITEMS)):
            pos = int(pos)
            if pos >= 0 and pos not in seen:
                chosen.append(pos)
                seen.add(pos)
                if len(chosen) == ANSWER_TOP:
                    break
    assert len(chosen) == len(set(map(int, chosen))) == ANSWER_TOP
    all_predictions.append([str(items.item_id.iat[int(pos)]) for pos in chosen])

answer_frame = pd.DataFrame({
    "query_id": benchmark_ids,
    "answer": [" ".join(row) for row in all_predictions],
})
temporary_csv = Path("/kaggle/working/answer_crossencoder_checked.csv")
answer_path = Path("/kaggle/working/answer.csv")
answer_frame.to_csv(temporary_csv, index=False, encoding="utf-8")

# Проверяем именно записанный CSV, включая множество допустимых item_id.
checked = pd.read_csv(temporary_csv, dtype=str, keep_default_na=False)
assert list(checked.columns) == ["query_id", "answer"]
assert checked.query_id.tolist() == benchmark_ids
allowed_ids = set(map(str, items.item_id))
for query_id, value in checked.itertuples(index=False, name=None):
    predicted = value.split()
    assert len(query_id) == 16 and len(predicted) == ANSWER_TOP
    assert len(set(predicted)) == ANSWER_TOP
    assert all(re.fullmatch(r"[0-9a-f]{16}", item_id)
               and item_id in allowed_ids for item_id in predicted)
os.replace(temporary_csv, answer_path)

metadata = {
    "pipeline": "pool_both -> CatBoost 1500 -> CatBoost 750 -> LambdaMART + BGE LoRA",
    "cross_rrf_weight": ANSWER_CROSS_WEIGHT,
    "query_sha256": benchmark_sha,
    "adapter_sha256": adapter_sha,
    "queries": N_BENCH,
    "changed_top50_vs_xgb_only": int(changed_vs_xgb),
    "fallback_queries": int(fallback_count),
    "cross_minutes": round(cross_minutes, 2),
}
(answer_path.parent / "answer_crossencoder_metadata.json").write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
print("Готово:", answer_path, "| размер:", answer_path.stat().st_size,
      "байт | изменились top-50:", changed_vs_xgb,
      "| дополнение до 50:", fallback_count,
      "| cross-encoder, мин:", round(cross_minutes, 1),
      "| весь расчёт, мин:", round((time.perf_counter()-_answer_t0)/60, 1))


## Приложение: подробное описание решения

# Поиск объявлений услуг Авито: отправленная версия

## Результат

Задача — выбрать для каждого из 2452 запросов до 50 объявлений из корпуса
189 212 объявлений. Целевая метрика — Recall@50: доля выбранных пользователем
объявлений, попавших в список кандидатов, усреднённая по запросам.

Отправленный `answer.csv` получил **Recall@50 = 0,850967** на проверяющей
системе. Это версия `avito_full_lambda_crossencoder_answer_kaggle.ipynb`:
BGE cross-encoder дообучался **три эпохи** с максимальной длиной пары
**192 токена**.

## Идея каскада

Тяжёлой моделью дорого оценивать каждый запрос со всеми 189 тысячами
объявлений. Каскад сокращает число пар постепенно. Ранние стадии рассчитаны
на полноту: пропущенное объявление следующий этап уже не восстановит.

| Этап | Размер на запрос | Назначение | Методы |
| --- | ---: | --- | --- |
| Весь корпус → широкий пул | Порядка 9 тыс., размер плавает | Собрать разные виды совпадений | SQLite FTS5/BM25, префиксы, география, LSA, обученные линейные векторы |
| Широкий пул → 1500, L1 | До 1500 | Сохранить высокую полноту | CatBoost QuerySoftMax с лексическими, семантическими, географическими и товарными признаками |
| 1500 → 750, L2 | До 750 | Точнее оценить пару | Второй CatBoost QuerySoftMax с дополнительными признаками заголовка, фильтров и описания |
| 750 → 50, L3 | 50 | Финальный выбор | XGBoost LambdaMART-topk и дообученный BGE-reranker-v2-m3; фиксированное объединение рангов |

Широкий пул — объединение источников, поэтому «10 тысяч» обозначает порядок
величины, а не точное число для каждого запроса. Лимиты 1500 и 750 выбраны
по Recall@K и стоимости следующего этапа. При промежуточном лимите 500
полнота была ниже; 750 оставляет больше подходящих объявлений для L3.
Ограничение задачи относится только к итоговым 50 объявлениям.

## Данные, признаки и модели

Положительные примеры — выбранные пары из `train.parquet`. Для поиска
используются `benchmark_queries.parquet` и
`benchmark_items.parquet`. У запроса учитываются текст, текст фильтров,
категория, локация и признак доставки. У объявления — заголовок, описание,
текст параметров, категория и подкатегория, локация и координаты, цена,
рейтинг, число отзывов и доступность контакта.

Лексический источник находит точные и почти точные названия услуг.
Географический источник добавляет местные и соседние объявления. LSA над
TF-IDF, линейные векторы, обученные на выбранных парах, и BGE-M3
обрабатывают переформулировки и синонимы. BGE-эмбеддинги объявлений
вычисляются на двух GPU. Локация и категория являются признаками, а не
безусловным исключением: подходящее объявление может находиться за
пределами выбранного города; `search_category=0` означает отсутствие
выбранной категории.

L1 получает BM25, ранги источников, косинусы LSA/линейной модели/BGE,
совпадение локации и расстояние, приоритет подкатегории, цену, рейтинг,
отзывы и совпадения слов. L2 добавляет признаки точной фразы и отдельных
слов в заголовке, параметрах и описании. LambdaMART обучен примерно на
10 000 группах запросов с построением пар у верхней части списка.

BGE-reranker-v2-m3 дообучен LoRA на 20 000 группах: для выбранного текста
берутся случайный, той же подкатегории и близкий по BGE кандидаты с разными
весами в парной функции потерь. Невыбранные объявления считаются *слабыми*
отрицательными примерами: неизвестно, видел ли их пользователь. Вход
cross-encoder — запрос, заголовок, параметры и начало описания; максимум
192 токена на пару. Результаты двух финальных моделей соединяются формулой
`0,85 / (60 + rank_LambdaMART) + 0,15 / (60 + rank_BGE)`.

Граф запросов, прямое запоминание `query_id → item_id` и бонус общей
популярности объявления в отправленной версии не используются.

## Проверка качества

Локальные запросы из train разделялись по тексту; одинаковая формулировка
не должна одновременно находиться в обучении и проверке supervised-этапа.
Оценка проводилась по выбранным объявлениям, которые есть в
`benchmark_items.parquet`. Recall измерялся на каждом пороге, чтобы
отделить потери генерации кандидатов от ошибок финального отбора.

| Измерение на отложенной группе из 500 запросов | Recall |
| --- | ---: |
| Широкий пул | 0,996 |
| L1 top-1500 | 0,994 |
| L2 top-750 | 0,986 |
| Отдельный LambdaMART top-50, опыт с 10 000 группами | 0,886 |
| Отправленный ансамбль, скрытая проверка | **0,850967** |

Последняя строка относится к **другим запросам**, поэтому её нельзя
непосредственно сравнивать вычитанием с локальными оценками. Локальная
группа уже просматривалась при разработке, так что результаты на ней
исследовательские. Перед отправкой CSV проверялся структурно: одна строка
на `query_id`, только колонки `query_id,answer`, 50 различных `item_id`
из корпуса в каждой строке.

## Анализ ошибок и изменения

1. **Нет общих слов при подходящей услуге.** Например, «барбер» и «мужская
   стрижка». К BM25 добавлены LSA, обученные линейные векторы и BGE-M3.
2. **Объявление находится в другой локации.** География используется как
   сигнал и дополнительный источник, но не как жёсткий фильтр.
3. **Услуга описана вне заголовка.** В L2 добавлены признаки описания;
   cross-encoder читает текст объявления. Обрезка до 192 токенов остаётся
   ограничением этой отправленной версии.
4. **Популярные объявления вытесняют объявления без истории.** В опытах
   полная история выборов увеличивала долю «тёплых» объявлений, но одна
   такая отправка дала лишь 0,815897. Общая популярность исключена из
   финальной версии.
5. **Основная потеря после top-750.** Локальный потолок L2 — 0,986, а
   отдельный LambdaMART top-50 — 0,886. Поэтому добавлен дообученный на
   услугах cross-encoder, но его влияние ограничено весом 0,15:
   чистый предобученный BGE работал существенно слабее модели с
   географическими и товарными признаками.

## Воспроизведение

Код этой версии находится в
`avito_full_lambda_crossencoder_answer_kaggle.ipynb`. Нужны три Parquet-файла,
Kaggle с двумя GPU T4 и доступ к весам `BAAI/bge-m3` и
`BAAI/bge-reranker-v2-m3`. Ноутбук выполняется сверху вниз и создаёт
`/kaggle/working/answer.csv`.
Используются открытые библиотеки NumPy, pandas, PyArrow, SciPy, scikit-learn,
CatBoost, XGBoost, PyTorch, Transformers, SentenceTransformers и PEFT.

> **Время выполнения:** полный прогон на Kaggle с **двумя GPU T4 займёт
> примерно 6 часов**. Это ориентир, а не гарантированный срок: время зависит
> от загрузки GPU, скорости чтения данных и повторного использования кеша.
> Сохраняйте контрольные точки и результаты долгих этапов, чтобы при
> прерывании сессии не пересчитывать их с начала.

SHA-256 отправленного файла:
`F164FCF9B5B75FE83532160C89154C91B5EC1ED572E09C02C7426B0C7226AC3A`.

## Статьи, документация и модели

Ниже перечислены источники методов. Их использование не означает
копирования готового решения задачи.

1. [Авито: «Как работает поисковое ранжирование для миллионов объявлений»](https://habr.com/ru/companies/avito/articles/846832/)
   — промышленный каскад поиска кандидатов, L1 и L2 ранжирование, признаки
   релевантности и действия пользователя.
2. [SQLite FTS5: полнотекстовый поиск и BM25](https://www.sqlite.org/fts5.html)
   — основа лексического источника и поиска по префиксам.
3. [Scikit-learn: TruncatedSVD и LSA](https://scikit-learn.org/stable/modules/decomposition.html#truncated-singular-value-decomposition-and-latent-semantic-analysis)
   — низкоразмерное представление текста поверх TF-IDF.
4. [BGE M3-Embedding: статья](https://arxiv.org/abs/2402.03216) и
   [карточка модели](https://huggingface.co/BAAI/bge-m3) — многоязычные
   плотные векторы для поиска и признаки семантического сходства.
5. [CatBoost: QuerySoftMax и другие функции ранжирования](https://catboost.ai/docs/en/concepts/loss-functions-ranking)
   — групповое обучение моделей L1 и L2.
6. [XGBoost: Learning to Rank](https://xgboost.readthedocs.io/en/stable/tutorials/learning_to_rank.html)
   — LambdaMART и построение пар с акцентом на верх выдачи.
7. [BAAI/bge-reranker-v2-m3](https://huggingface.co/BAAI/bge-reranker-v2-m3)
   — предобученная многоязычная модель парной оценки текста для L3.
8. [LoRA: Low-Rank Adaptation of Large Language Models](https://arxiv.org/abs/2106.09685)
   — параметрически экономное дообучение cross-encoder.
9. [Reciprocal Rank Fusion Outperforms Condorcet and Individual Rank Learning Methods](https://doi.org/10.1145/1571941.1572114)
   — объединение рангов LambdaMART и cross-encoder.
